# Retail credit risk lifecycle on ONE book: PD, term structure, LGD, EAD, IFRS 9 staging and ECL

**Book.** LendingClub accepted consumer loans, 2007-2018Q4 (public Kaggle dataset `wordsforthewise/lending-club`).
Every number below comes from this single dataset. There is no second portfolio and no blended figure.

**What this notebook does.** It walks the usual retail-credit-risk development path:
framing and data audit, 12-month PD ladder (logistic, WoE scorecard, forests, boosting), frozen champion rule,
calibration on validation only, backtest of predicted vs actual defaults, PD term structure, EAD, an honest LGD study
(including the possibility that nothing beats the mean), IFRS 9 stage allocation and expected credit loss (ECL) on the
live book, monitoring and a cut-off illustration.

**Plain-English summary.** We learn from old loans which application details go with people who stop paying within a
year, test the result on a later year the model never saw, and then use it to estimate how much money a lender
should set aside today for the loans still on its books.

**What changed versus the first version of this project (audit fixes).**
1. The "champion beat the scorecard, DeLong p = 0.002" claim compared CatBoost with LightGBM. It is recomputed here against the WoE scorecard.
2. "LGD 0.902, validation r = 0.993" mixed two things: 0.902 is the *mean* LGD and r = 0.993 validated the *balance amortisation* (EAD), not LGD. LGD is now modelled honestly against the mean on an out-of-time cohort.
3. Calibration ECE is reported together with the backtest (predicted vs actual default rate).
4. The label dropped every loan still `Current`, which inflated the out-of-time bad rate (17.3% vs a true ~6%) and the data snapshot was taken as Dec-2018 although the file is Mar-2019. Both are fixed.
5. One dataset end to end (LendingClub). The Home Credit staging module and the macro/reject-inference add-ons are removed.

## Cell guide
0 setup and library | 1 load and audit | 2 framing, labels, duplicates, leakage | 3 split and data checks |
4 univariate WoE/IV | 5 multicollinearity | 6 model ladder and tuning | 7 frozen champion rule | 8 out-of-time scoring and significance |
9 calibration (validation only) | 10 backtest and master scale | 11 scorecard points | 12 term structure | 13 EAD |
14 LGD | 15 live book, SICR and stages | 16 ECL | 17 monitoring | 18 cut-off illustration | 19 save artifacts

### Cell 0 - library (written to disk so the notebook is self-contained)

In [1]:
%%writefile rcl_lib.py
"""rcl_lib - shared code for the RetailCreditLifecycle notebook (LendingClub only).

ONE dataset end to end: LendingClub accepted loans 2007-2018Q4 (public Kaggle dataset
wordsforthewise/lending-club).  PD, term structure, LGD, EAD, staging and ECL are all
built on this same book; nothing is borrowed from another dataset.

Everything a stage needs lives here so the notebook cells stay short and readable.
Fixed seeds, no randomness outside seeded paths, ASCII-safe prints.
"""
from __future__ import annotations

import json
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (auc as sk_auc, average_precision_score, brier_score_loss,
                             log_loss, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
# The "2018Q4" file was pulled in March 2019: last_pymnt_d of Current loans is Mar-2019 and
# last_credit_pull_d is Mar-2019.  The snapshot month is therefore READ FROM THE DATA
# (max last_pymnt_d), not assumed to be Dec-2018 (the old project's 3-month age error).
SNAP_IDX = 2019 * 12 + 3
HORIZON_M = 12                             # PD horizon (months from issue)
LAST_PAY_MAX = 9                           # stopped paying by month 9 => 90+ DPD inside 12 months
MIN_AGE = 15                               # 12m window + ~3m to reach 90 DPD + charge-off posting lag
CO_LAG_M = 4                               # est. charge-off date = last payment + 4 months (120 DPD)

PD_FEATURES = [
    "loan_amnt", "term_months", "emp_length_years", "annual_inc", "dti",
    "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
    "revol_util", "total_acc", "mort_acc", "fico_mid", "purpose", "addr_state",
    "home_ownership", "verification_status", "application_type",
]
CATEGORICAL = ["purpose", "addr_state", "home_ownership", "verification_status",
               "application_type"]
NUMERIC = [c for c in PD_FEATURES if c not in CATEGORICAL]
SCALING = {"base_score": 600, "pdo": 20, "base_odds": 20.0}

# columns that are known only AFTER origination (never PD features)
LEAKAGE_FIELDS = [
    "out_prncp", "total_rec_prncp", "total_rec_int", "total_pymnt", "recoveries",
    "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low",
    "loan_status", "settlement_status", "debt_settlement_flag", "months_on_book",
    "months_to_last_pymnt",
    "grade", "sub_grade", "int_rate", "installment",   # LendingClub's own price/risk output
]

RAW_COLS = [
    "id", "loan_amnt", "funded_amnt", "term", "int_rate", "installment", "grade",
    "sub_grade", "emp_length", "home_ownership", "annual_inc", "verification_status",
    "issue_d", "loan_status", "purpose", "addr_state", "dti", "delinq_2yrs",
    "fico_range_low", "fico_range_high", "inq_last_6mths", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc", "mort_acc", "out_prncp", "total_rec_prncp",
    "total_rec_int", "total_pymnt", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "application_type", "last_fico_range_high",
    "last_fico_range_low", "last_credit_pull_d",
]
NUM_RAW = ["id", "loan_amnt", "funded_amnt", "installment", "annual_inc", "dti",
           "delinq_2yrs", "fico_range_low", "fico_range_high", "inq_last_6mths",
           "open_acc", "pub_rec", "revol_bal", "total_acc", "mort_acc", "out_prncp",
           "total_rec_prncp", "total_rec_int", "total_pymnt", "recoveries",
           "collection_recovery_fee", "last_pymnt_amnt", "last_fico_range_high",
           "last_fico_range_low"]
CAT_RAW = ["grade", "sub_grade", "home_ownership", "verification_status", "purpose",
           "addr_state", "loan_status", "application_type"]

# defaults seen by the lender: charged off / defaulted / 31-120 days late
EVENT_STATUS = ["Charged Off", "Default", "Late (31-120 days)",
                "Does not meet the credit policy. Status:Charged Off"]
CHARGED_OFF_STATUS = ["Charged Off", "Does not meet the credit policy. Status:Charged Off"]
ACTIVE_STATUS = ["Current", "Late (16-30 days)", "Late (31-120 days)", "In Grace Period",
                 "Default"]

_T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.time() - _T0:7.0f}s] {msg}", flush=True)


# --------------------------------------------------------------------------------------
# data loading
# --------------------------------------------------------------------------------------
def month_index(dt: pd.Series) -> pd.Series:
    return dt.dt.year * 12 + dt.dt.month


def _pct(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.astype(str).str.strip().str.rstrip("%"), errors="coerce")


def _parse_chunk(c: pd.DataFrame) -> pd.DataFrame:
    for col in NUM_RAW:
        c[col] = pd.to_numeric(c[col], errors="coerce")
    c["issue_d"] = pd.to_datetime(c["issue_d"], format="%b-%Y", errors="coerce")
    c["last_pymnt_d"] = pd.to_datetime(c["last_pymnt_d"], format="%b-%Y", errors="coerce")
    c["last_credit_pull_d"] = pd.to_datetime(c["last_credit_pull_d"], format="%b-%Y",
                                             errors="coerce")
    c["int_rate"] = _pct(c["int_rate"])
    c["revol_util"] = _pct(c["revol_util"])
    c["term_months"] = pd.to_numeric(c["term"].astype(str).str.extract(r"(\d+)",
                                                                       expand=False),
                                     errors="coerce")
    emp = c["emp_length"].astype(str).str.strip()
    ey = pd.Series(np.nan, index=c.index, dtype="float64")
    ey[emp == "< 1 year"] = 0.0
    parsed = pd.to_numeric(emp.str.extract(r"^(\d+)\+? years?$", expand=False),
                           errors="coerce")
    ey[parsed.notna()] = parsed
    c["emp_length_years"] = ey
    c["fico_mid"] = (c["fico_range_low"] + c["fico_range_high"]) / 2.0
    c["last_fico_mid"] = (c["last_fico_range_low"] + c["last_fico_range_high"]) / 2.0
    for col in CAT_RAW:
        c[col] = c[col].astype("category")
    return c.drop(columns=["term", "emp_length"])


def load_accepted(path, sample_frac: float | None = None, chunk: int = 400_000,
                  seed: int = SEED):
    """Read the raw LendingClub accepted file chunk by chunk (memory-safe).

    Returns (frame, audit).  Footer/summary rows (no loan_amnt / issue_d) are dropped and
    counted.  `sample_frac` keeps a seeded random share of rows (FAST/smoke mode only).
    """
    rng = np.random.default_rng(seed)
    parts, audit = [], {"raw_rows": 0, "footer_rows_dropped": 0}
    reader = pd.read_csv(path, usecols=lambda c: c in set(RAW_COLS), dtype=str,
                         low_memory=False, chunksize=chunk)
    for raw in reader:
        audit["raw_rows"] += len(raw)
        parsed = _parse_chunk(raw)
        footer = parsed["loan_amnt"].isna() | parsed["issue_d"].isna() | parsed["id"].isna()
        audit["footer_rows_dropped"] += int(footer.sum())
        parsed = parsed[~footer]
        if sample_frac and sample_frac < 1.0:
            parsed = parsed[rng.random(len(parsed)) < sample_frac]
        parts.append(parsed)
    frame = pd.concat(parts, ignore_index=True)
    for col in CAT_RAW:   # categories must be consistent after concat
        frame[col] = frame[col].astype("category")
    frame["id"] = frame["id"].astype("int64")
    audit["rows_kept"] = int(len(frame))
    audit["sampled"] = bool(sample_frac and sample_frac < 1.0)
    return frame, audit


def snapshot_index(df: pd.DataFrame) -> int:
    """Snapshot month index = latest last-payment month in the file."""
    return int(month_index(df["last_pymnt_d"].dropna()).max())


def derive_basics(df: pd.DataFrame, snap_idx: int | None = None) -> pd.DataFrame:
    """Time fields measured against the data snapshot (Mar-2019, read from the file)."""
    df = df.copy()
    snap_idx = snap_idx or snapshot_index(df)
    df.attrs["snap_idx"] = snap_idx
    df["issue_month"] = df["issue_d"].dt.strftime("%Y-%m")
    df["mob"] = (snap_idx - month_index(df["issue_d"])).astype("int32")
    df["mtlp"] = month_index(df["last_pymnt_d"]) - month_index(df["issue_d"])  # months to last payment
    return df


def remove_duplicates(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Duplicates are removed BEFORE any split so the same loan can never sit in two splits."""
    n0 = len(df)
    dup_id = int(df["id"].duplicated().sum())
    df = df.drop_duplicates(subset="id", keep="first")
    keys = ["issue_d", "loan_amnt", "funded_amnt", "term_months", "annual_inc", "dti",
            "fico_mid", "revol_bal", "revol_util", "open_acc", "total_acc", "addr_state",
            "purpose", "emp_length_years", "inq_last_6mths", "delinq_2yrs"]
    dup_app = int(df.duplicated(subset=keys, keep="first").sum())
    df = df.drop_duplicates(subset=keys, keep="first")
    return df.reset_index(drop=True), {"rows_in": n0, "duplicate_ids_removed": dup_id,
                                       "duplicate_applications_removed": dup_app,
                                       "rows_out": int(len(df))}


# --------------------------------------------------------------------------------------
# labels and splits
# --------------------------------------------------------------------------------------
def add_labels(df: pd.DataFrame) -> pd.DataFrame:
    """12-month PD label (frozen definition, see notebook cell 'Framing').

    bad_12m = 1 : loan ends charged off / defaulted / 31-120 days late AND its last payment
                 was made in month <= 9 (so it was 90+ days past due inside the first 12
                 months).  A loan that never paid anything (last_pymnt_d missing) is a
                 first-payment default and counts as bad.
    bad_12m = 0 : every other loan old enough to be observed (age >= 15 months),
                 INCLUDING loans still Current - they survived the window.
    undefined   : age < 15 months (window or charge-off posting lag not complete), or a
                 still-active loan that stopped paying inside the window (indeterminate).
    """
    df = df.copy()
    status = df["loan_status"].astype(str)
    event = status.isin(EVENT_STATUS)
    mtlp = df["mtlp"].astype(float)
    never_paid = event & mtlp.isna()
    mtlp_eff = mtlp.where(~never_paid, 0.0)
    stopped_in_window = mtlp_eff <= LAST_PAY_MAX
    active_nonevent = status.isin(["Current", "Late (16-30 days)", "In Grace Period"])
    indeterminate = active_nonevent & stopped_in_window & mtlp.notna()
    seasoned = df["mob"] >= MIN_AGE
    bad = (event & stopped_in_window).astype(float)
    pop = seasoned & ~indeterminate
    df["is_event"] = event
    df["never_paid"] = never_paid
    df["resolved_12m"] = pop
    df["bad_12m"] = np.where(pop, bad, np.nan)
    # legacy definition (for the comparison table only; NOT used for modelling)
    co = status.isin(["Charged Off", "Default"])
    fp = status.isin(["Fully Paid"])
    old_bad = co & (mtlp.fillna(np.inf) <= 12)
    old_good = fp & (df["mob"] >= 12)
    df["bad_12m_legacy"] = np.where(old_bad, 1.0, np.where(old_good, 0.0, np.nan))
    # default time (month in which the loan first reaches 90 DPD) for hazard work
    df["t_default"] = np.where(event, mtlp_eff + 3.0, np.nan)
    return df


def add_split(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    y = df["issue_d"].dt.year
    df["split"] = np.select(
        [df["issue_d"] <= pd.Timestamp("2015-12-31"), y == 2016, y == 2017, y >= 2018],
        ["train", "valid", "oot", "psi_only"], default="unassigned")
    return df


# --------------------------------------------------------------------------------------
# metrics
# --------------------------------------------------------------------------------------
def gini(y, s) -> float:
    return float(2.0 * roc_auc_score(y, s) - 1.0)


def ks_stat(y, s) -> float:
    """KS evaluated at distinct score values (tie-safe)."""
    y = np.asarray(y, float)
    g = (pd.DataFrame({"y": y, "s": np.asarray(s, float)}).groupby("s", sort=True)["y"]
         .agg(["sum", "count"]))
    cb = g["sum"].cumsum() / y.sum()
    cg = (g["count"] - g["sum"]).cumsum() / (len(y) - y.sum())
    return float(np.max(np.abs(cb.to_numpy() - cg.to_numpy())))


def lift_at(y, s, frac=0.10) -> float:
    y = np.asarray(y)
    k = int(round(len(y) * frac))
    top = np.argsort(-np.asarray(s))[:k]
    return float(y[top].mean() / y.mean()) if y.mean() else float("nan")


def pr_auc(y, s) -> float:
    """Average precision (step-wise area under the precision-recall curve)."""
    return float(average_precision_score(y, s))


def ece_and_deciles(y, p, n_bins=10):
    f = pd.DataFrame({"y": np.asarray(y, float), "p": np.asarray(p, float)})
    f["d"] = pd.qcut(f["p"].rank(method="first"), n_bins, labels=False)
    rows, ece = [], 0.0
    for d, g in f.groupby("d"):
        w = len(g) / len(f)
        e, a = float(g["p"].mean()), float(g["y"].mean())
        rows.append({"decile": int(d) + 1, "n": int(len(g)), "expected_pd": round(e, 5),
                     "actual_pd": round(a, 5), "difference": round(a - e, 5)})
        ece += w * abs(a - e)
    return float(ece), rows


def ece_point(y, p, n_bins=10) -> float:
    return ece_and_deciles(y, p, n_bins)[0]


def evaluate(y, p) -> dict:
    return {"n": int(len(y)), "bad_rate": round(float(np.mean(y)), 5),
            "auc": round(float(roc_auc_score(y, p)), 5), "gini": round(gini(y, p), 5),
            "ks": round(ks_stat(y, p), 5), "pr_auc": round(pr_auc(y, p), 5),
            "brier": round(float(brier_score_loss(y, p)), 5),
            "log_loss": round(float(log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))), 5),
            "lift_10": round(lift_at(y, p, 0.10), 4)}


def _midrank(v):
    order = np.argsort(v)
    z = v[order]
    n = len(v)
    t = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1.0
        i = j
    out = np.empty(n)
    out[order] = t
    return out


def delong_test(y, a, b) -> dict:
    """Paired DeLong test (Sun & Xu 2014) for two correlated AUCs on the same rows."""
    y = np.asarray(y).astype(int)
    sc = np.vstack([np.asarray(a, float), np.asarray(b, float)])
    order = np.argsort(-y, kind="mergesort")
    m = int(y.sum())
    ss = sc[:, order]
    n = ss.shape[1] - m
    pos, neg = ss[:, :m], ss[:, m:]
    tx, ty, tz = np.empty((2, m)), np.empty((2, n)), np.empty((2, m + n))
    for r in range(2):
        tx[r], ty[r], tz[r] = _midrank(pos[r]), _midrank(neg[r]), _midrank(ss[r])
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    d = aucs[0] - aucs[1]
    if var <= 0:
        z, p, se = 0.0, 1.0, 0.0
    else:
        se = math.sqrt(var)
        z = d / se
        p = 2.0 * float(stats.norm.sf(abs(z)))
    return {"auc_a": round(float(aucs[0]), 5), "auc_b": round(float(aucs[1]), 5),
            "delta_auc": round(float(d), 5), "delta_gini": round(float(2 * d), 5),
            "delta_gini_ci95_delong": [round(float(2 * (d - 1.96 * se)), 5),
                                       round(float(2 * (d + 1.96 * se)), 5)],
            "z": round(float(z), 4), "p_value": float(p)}


def bootstrap_gini_delta(y, a, b, n_iter=1000, seed=SEED) -> dict:
    """Paired row bootstrap of Gini(a) - Gini(b)."""
    rng = np.random.default_rng(seed)
    y, a, b = np.asarray(y), np.asarray(a), np.asarray(b)
    idx = np.arange(len(y))
    d = np.full(n_iter, np.nan)
    for i in range(n_iter):
        s = rng.choice(idx, size=len(idx), replace=True)
        if y[s].sum() in (0, len(s)):
            continue
        d[i] = gini(y[s], a[s]) - gini(y[s], b[s])
    k = ~np.isnan(d)
    return {"delta": round(float(np.nanmean(d)), 5),
            "ci95": [round(float(np.percentile(d[k], 2.5)), 5),
                     round(float(np.percentile(d[k], 97.5)), 5)],
            "p_two_sided": round(float(2 * min((d[k] <= 0).mean(), (d[k] >= 0).mean())), 5),
            "n_iter": int(k.sum())}


def psi(expected, actual, n_bins=10) -> float:
    expected, actual = np.asarray(expected, float), np.asarray(actual, float)
    edges = np.unique(np.quantile(expected[~np.isnan(expected)], np.linspace(0, 1, n_bins + 1)))
    if len(edges) < 3:
        return float("nan")
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.clip(np.histogram(expected[~np.isnan(expected)], edges)[0] / np.sum(~np.isnan(expected)), 1e-6, None)
    a = np.clip(np.histogram(actual[~np.isnan(actual)], edges)[0] / np.sum(~np.isnan(actual)), 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))


def psi_band(v: float) -> str:
    return "unknown" if not np.isfinite(v) else ("stable" if v < 0.10 else ("watch" if v <= 0.25 else "shift"))


def score_from_pd(p):
    f = SCALING["pdo"] / math.log(2.0)
    off = SCALING["base_score"] - f * math.log(SCALING["base_odds"])
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return off - f * np.log(p / (1 - p))


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return [jsonable(v) for v in o.tolist()]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        v = float(o)
        return v if np.isfinite(v) else None
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (pd.Timestamp,)):
        return str(o.date())
    return o


def write_json(obj, path: Path) -> None:
    Path(path).write_text(json.dumps(jsonable(obj), indent=2, sort_keys=False, default=str),
                          encoding="utf-8")


# --------------------------------------------------------------------------------------
# univariate WoE / IV and multicollinearity
# --------------------------------------------------------------------------------------
class WOETransformer(BaseEstimator, TransformerMixin):
    """Supervised WoE encoding, quantile bins for numerics, categories as-is.
    Fitted inside the pipeline (so inside every CV fold), never on validation/test."""

    def __init__(self, n_bins: int = 8):
        self.n_bins = n_bins

    def _keys(self, col, values):
        if col in self.edges_:
            e = self.edges_[col]
            v = pd.to_numeric(values, errors="coerce").to_numpy(float)
            k = np.searchsorted(e[1:-1], v, side="right").astype(int)
            k[np.isnan(v)] = -1
            return pd.Series(k, index=values.index)
        return values.astype(str)

    def fit(self, frame, y):
        y = np.asarray(y, float)
        self.columns_ = list(frame.columns)
        self.maps_, self.edges_ = {}, {}
        for col in self.columns_:
            v = frame[col]
            if pd.api.types.is_numeric_dtype(v):
                vals = v.dropna()
                e = np.unique(np.quantile(vals, np.linspace(0, 1, self.n_bins + 1)))
                if len(e) >= 3:
                    self.edges_[col] = e
            keys = self._keys(col, v)
            t = pd.DataFrame({"k": keys.to_numpy(), "y": y}).groupby("k")["y"].agg(["sum", "count"])
            good = t["count"] - t["sum"]
            pb = (t["sum"] + 0.5) / (t["sum"] + 0.5).sum()
            pg = (good + 0.5) / (good + 0.5).sum()
            self.maps_[col] = np.log(pb / pg).to_dict()
        return self

    def transform(self, frame):
        out = {}
        for col in self.columns_:
            m = self.maps_[col]
            fb = float(np.mean(list(m.values())))
            out[col] = self._keys(col, frame[col]).map(m).fillna(fb).to_numpy(float)
        return pd.DataFrame(out, index=frame.index)

    def bin_label(self, col, key):
        if col not in self.edges_:
            return str(key)
        if key == -1:
            return "missing"
        e = self.edges_[col]
        lo = "-inf" if key == 0 else f"{e[key]:.4g}"
        hi = "+inf" if key >= len(e) - 2 else f"{e[key + 1]:.4g}"
        return f"[{lo}, {hi})"


def iv_table(x: pd.DataFrame, y, n_bins=10) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Information value per feature + the bin-level WoE table (training data only)."""
    w = WOETransformer(n_bins=n_bins).fit(x, y)
    y = np.asarray(y, float)
    rows, bins = [], []
    for col in w.columns_:
        keys = w._keys(col, x[col]).to_numpy()
        t = pd.DataFrame({"k": keys, "y": y}).groupby("k")["y"].agg(["sum", "count"])
        good = t["count"] - t["sum"]
        pb = (t["sum"] + 0.5) / (t["sum"] + 0.5).sum()
        pg = (good + 0.5) / (good + 0.5).sum()
        woe = np.log(pb / pg)
        iv = float(((pb - pg) * woe).sum())
        rows.append({"feature": col, "iv": iv,
                     "strength": ("suspect (>0.5)" if iv > 0.5 else "strong" if iv > 0.3
                                  else "medium" if iv > 0.1 else "weak" if iv >= 0.02
                                  else "useless (<0.02)")})
        if col in w.edges_ or x[col].nunique() <= 12:
            for k in t.index:
                bins.append({"feature": col, "bin": w.bin_label(col, k), "n": int(t.loc[k, "count"]),
                             "bad_rate": float(t.loc[k, "sum"] / t.loc[k, "count"]),
                             "woe": float(woe.loc[k])})
    return pd.DataFrame(rows).sort_values("iv", ascending=False).reset_index(drop=True), pd.DataFrame(bins)


def vif_table(x_num: pd.DataFrame, max_rows=100_000, drop_above=5.0):
    """VIF on median-imputed numerics, then iterative drop of the worst feature above 5."""
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    d = x_num.sample(min(len(x_num), max_rows), random_state=SEED)
    d = d.fillna(d.median())
    d = (d - d.mean()) / d.std().replace(0, 1)
    cols = list(d.columns)
    first = None
    dropped = []
    while True:
        arr = d[cols].to_numpy()
        v = pd.Series([variance_inflation_factor(arr, i) for i in range(len(cols))], index=cols)
        if first is None:
            first = v.copy()
        if v.max() <= drop_above or len(cols) <= 2:
            break
        worst = v.idxmax()
        dropped.append({"feature": worst, "vif_at_drop": float(v.max())})
        cols.remove(worst)
    out = pd.DataFrame({"feature": first.index, "vif_initial": first.values})
    out["dropped_in_iteration"] = out["feature"].isin([r["feature"] for r in dropped])
    return out.sort_values("vif_initial", ascending=False).reset_index(drop=True), dropped


# --------------------------------------------------------------------------------------
# PD models
# --------------------------------------------------------------------------------------
class SimplePrep(BaseEstimator, TransformerMixin):
    """Median-impute numerics, integer-code categoricals (unseen -> -1), plain dict params."""

    def __init__(self, numeric=None, categorical=None, min_category_freq: int = 50):
        self.numeric = numeric
        self.categorical = categorical
        self.min_category_freq = min_category_freq

    def fit(self, frame, y=None):
        self.medians_ = {c: float(pd.to_numeric(frame[c], errors="coerce").median())
                         for c in self.numeric}
        self.category_codes_ = {}
        for col in self.categorical:
            counts = frame[col].astype(str).value_counts()
            levels = sorted(counts[counts >= self.min_category_freq].index)
            self.category_codes_[col] = {lv: i for i, lv in enumerate(levels)}
        return self

    def transform(self, frame):
        n, k = len(frame), len(self.numeric)
        out = np.zeros((n, k + len(self.categorical)), dtype=np.float32)
        for j, col in enumerate(self.numeric):
            out[:, j] = pd.to_numeric(frame[col], errors="coerce").astype(float).fillna(
                self.medians_[col]).to_numpy(np.float32)
        for j, col in enumerate(self.categorical):
            out[:, k + j] = frame[col].astype(str).map(self.category_codes_[col]).fillna(-1).to_numpy(np.float32)
        return pd.DataFrame(out, columns=list(self.numeric) + list(self.categorical), index=frame.index)


def one_hot_prep():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler(with_mean=False))]), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                   min_frequency=0.01, sparse_output=True))]),
         CATEGORICAL)], sparse_threshold=1.0)


def tree_pipeline(est, min_category_freq=50):
    return Pipeline([("prep", SimplePrep(NUMERIC, CATEGORICAL, min_category_freq)), ("clf", est)])


def woe_pipeline(clf):
    return Pipeline([("woe", WOETransformer()), ("clf", clf)])


def build_candidates(fast: bool = False) -> list:
    import lightgbm
    import xgboost
    import catboost
    it = 60 if fast else 400
    nj = 4
    C = []
    C.append({"name": "LogisticRegression L2 (one-hot, tuned C)", "family": "classical linear",
              "pipeline": Pipeline([("prep", one_hot_prep()),
                                    ("clf", LogisticRegression(max_iter=3000, random_state=SEED))]),
              "space": {"clf__C": [0.01, 0.1, 1.0, 10.0]}, "kind": "grid", "n_iter": None})
    C.append({"name": "WOE logistic scorecard (tuned C x bins)", "family": "credit scorecard",
              "pipeline": woe_pipeline(LogisticRegression(max_iter=3000, random_state=SEED)),
              "space": {"woe__n_bins": [6, 10], "clf__C": [0.01, 0.1, 1.0, 10.0]},
              "kind": "grid", "n_iter": None})
    C.append({"name": "WOE L1 logistic (lasso, tuned C)", "family": "classical linear",
              "pipeline": woe_pipeline(LogisticRegression(penalty="l1", solver="liblinear",
                                                          max_iter=3000, random_state=SEED)),
              "space": {"clf__C": [0.01, 0.05, 0.2, 1.0]}, "kind": "grid", "n_iter": None})
    C.append({"name": "RandomForest (tuned)", "family": "tree ensemble",
              "pipeline": tree_pipeline(RandomForestClassifier(random_state=SEED, n_jobs=nj)),
              "space": {"clf__n_estimators": [150, 250], "clf__max_depth": [8, 12, 18, None],
                        "clf__min_samples_leaf": [20, 50, 150], "clf__max_features": ["sqrt", 0.3],
                        "clf__class_weight": [None, "balanced_subsample"]},
              "kind": "random", "n_iter": 3 if fast else 6})
    C.append({"name": "LightGBM (tuned)", "family": "tree ensemble",
              "pipeline": tree_pipeline(lightgbm.LGBMClassifier(random_state=SEED, n_jobs=nj,
                                                                verbosity=-1, n_estimators=it)),
              "space": {"clf__n_estimators": [200, 400], "clf__learning_rate": [0.03, 0.06, 0.1],
                        "clf__num_leaves": [15, 31, 63], "clf__min_child_samples": [50, 200, 500],
                        "clf__subsample": [0.7, 0.9], "clf__colsample_bytree": [0.7, 0.9],
                        "clf__reg_alpha": [0.0, 0.5], "clf__reg_lambda": [0.0, 1.0, 5.0]},
              "kind": "random", "n_iter": 3 if fast else 15})
    C.append({"name": "XGBoost (tuned)", "family": "tree ensemble",
              "pipeline": tree_pipeline(xgboost.XGBClassifier(random_state=SEED, n_jobs=nj,
                                                              tree_method="hist", eval_metric="logloss",
                                                              n_estimators=it)),
              "space": {"clf__n_estimators": [200, 400], "clf__learning_rate": [0.03, 0.06, 0.1],
                        "clf__max_depth": [3, 5, 7], "clf__min_child_weight": [1, 5, 20],
                        "clf__subsample": [0.7, 0.9], "clf__colsample_bytree": [0.7, 0.9],
                        "clf__reg_alpha": [0.0, 0.5], "clf__reg_lambda": [1.0, 5.0],
                        "clf__gamma": [0.0, 1.0]},
              "kind": "random", "n_iter": 3 if fast else 12})
    C.append({"name": "CatBoost (tuned)", "family": "tree ensemble",
              "pipeline": tree_pipeline(catboost.CatBoostClassifier(random_seed=SEED, verbose=0,
                                                                    allow_writing_files=False,
                                                                    thread_count=nj, iterations=it)),
              "space": {"clf__iterations": [200, 400], "clf__learning_rate": [0.03, 0.06, 0.1],
                        "clf__depth": [4, 6, 8], "clf__l2_leaf_reg": [1.0, 3.0, 10.0],
                        "clf__random_strength": [0.5, 1.0], "clf__bagging_temperature": [0.0, 0.5]},
              "kind": "random", "n_iter": 3 if fast else 10})
    return C


def search_and_refit(cand, x_search, y_search, cv, x_train, y_train):
    """One CV scheme for every tunable family (roc_auc); the best config is refit on full train."""
    t0 = time.time()
    if cand["kind"] == "grid":
        s = GridSearchCV(cand["pipeline"], cand["space"], cv=cv, scoring="roc_auc", refit=False, n_jobs=1)
        n_cand = int(np.prod([len(v) for v in cand["space"].values()]))
    else:
        s = RandomizedSearchCV(cand["pipeline"], cand["space"], n_iter=cand["n_iter"], cv=cv,
                               scoring="roc_auc", refit=False, n_jobs=1, random_state=SEED)
        n_cand = int(cand["n_iter"])
    s.fit(x_search, y_search)
    best = {k: (v.item() if isinstance(v, np.generic) else v) for k, v in s.best_params_.items()}
    t_search = time.time() - t0
    model = clone(cand["pipeline"]).set_params(**best)
    t1 = time.time()
    model.fit(x_train, y_train)
    info = {"search": "GridSearchCV" if cand["kind"] == "grid" else "RandomizedSearchCV",
            "n_candidates": n_cand, "n_folds": cv.get_n_splits(), "best_cv_roc_auc": round(float(s.best_score_), 5),
            "best_cv_gini": round(2 * float(s.best_score_) - 1, 5), "best_params": best,
            "search_seconds": round(t_search, 1), "refit_seconds": round(time.time() - t1, 1)}
    log(f"  {cand['name']}: CV auc={s.best_score_:.4f} ({n_cand}x{cv.get_n_splits()} fits, {t_search:.0f}s) {best}")
    return model, info


# --------------------------------------------------------------------------------------
# calibration (fitted on VALIDATION only; own implementation = no hidden refits)
# --------------------------------------------------------------------------------------
class Calibrator:
    """Maps a raw model probability to a calibrated PD.  method in {raw, isotonic, platt}."""

    def __init__(self, method: str):
        self.method = method

    def fit(self, p_raw, y):
        p_raw = np.asarray(p_raw, float)
        if self.method == "isotonic":
            self.m_ = IsotonicRegression(out_of_bounds="clip", y_min=1e-5, y_max=1 - 1e-5).fit(p_raw, y)
        elif self.method == "platt":
            lg = np.log(np.clip(p_raw, 1e-6, 1 - 1e-6) / (1 - np.clip(p_raw, 1e-6, 1 - 1e-6)))
            self.m_ = LogisticRegression(C=1e6, max_iter=1000).fit(lg.reshape(-1, 1), y)
        return self

    def predict(self, p_raw):
        p_raw = np.asarray(p_raw, float)
        if self.method == "raw":
            return p_raw
        if self.method == "isotonic":
            return self.m_.predict(p_raw)
        lg = np.log(np.clip(p_raw, 1e-6, 1 - 1e-6) / (1 - np.clip(p_raw, 1e-6, 1 - 1e-6)))
        return self.m_.predict_proba(lg.reshape(-1, 1))[:, 1]


def cv_ece_of_calibrator(method, p_raw, y, n_splits=5, seed=SEED):
    """Cross-validated ECE of a calibrator fitted on part of VALID and scored on the rest."""
    p_raw, y = np.asarray(p_raw, float), np.asarray(y, float)
    out = np.zeros(len(y))
    for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(p_raw, y):
        out[te] = Calibrator(method).fit(p_raw[tr], y[tr]).predict(p_raw[te])
    return ece_point(y, out)


def binom_row(k: int, n: int, p_pred: float) -> dict:
    """Exact two-sided binomial test of observed defaults vs predicted PD (+ Wilson CI)."""
    if n == 0:
        return {"binom_p": None, "ci95_actual": [None, None]}
    p = float(stats.binomtest(int(k), int(n), float(np.clip(p_pred, 1e-9, 1 - 1e-9))).pvalue)
    ci = stats.binomtest(int(k), int(n)).proportion_ci(confidence_level=0.95, method="wilson")
    return {"binom_p": p, "ci95_actual": [float(ci.low), float(ci.high)]}


# --------------------------------------------------------------------------------------
# term structure (Kaplan-Meier default hazard + cumulative incidence with prepayment)
# --------------------------------------------------------------------------------------
def hazard_by_month(df: pd.DataFrame, max_m: int = 84, min_at_risk: int = 200) -> dict:
    """Exposure-based monthly default hazard by month on book.

    Event time = month the loan reaches 90 DPD (months_to_last_payment + 3).  Loans leave the
    risk set when they are paid off (competing risk) or when the file ends (administrative
    censoring at their age).  h_k = defaults in month k / loans still at risk at the start
    of month k.  `cif` = cumulative incidence of default allowing for prepayment;
    `km` = default-only cumulative (as if nobody prepaid).  Months where fewer than
    `min_at_risk` loans remain at risk get hazard 0 (tail of the observed curve).
    """
    ev = df["is_event"].to_numpy(bool)
    t_def = df["t_default"].to_numpy(float)
    status = df["loan_status"].astype(str).to_numpy()
    fully = np.isin(status, ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"])
    t_pre = df["mtlp"].to_numpy(float)
    mob = df["mob"].to_numpy(float)
    t_cens = np.where(fully, t_pre, mob)
    t_cens = np.where(np.isnan(t_cens), 0, t_cens)
    t_def = np.where(np.isnan(t_def), 0, t_def)

    def cnt(x, mask):
        x = np.clip(np.floor(x[mask]).astype(int), 0, max_m)
        return np.bincount(x, minlength=max_m + 1)[: max_m + 1]

    d = cnt(t_def, ev)
    pre = cnt(t_cens, (~ev) & fully)
    cen = cnt(t_cens, (~ev) & (~fully))
    n = len(df)
    exits_before = np.concatenate([[0], np.cumsum(d + pre + cen)[:-1]])
    at_risk = n - exits_before
    with np.errstate(divide="ignore", invalid="ignore"):
        h = np.where(at_risk > 0, d / at_risk, 0.0)
        hp = np.where(at_risk > 0, pre / at_risk, 0.0)
    h[0] = 0.0
    hp[0] = 0.0
    thin = at_risk < min_at_risk          # a handful of survivors makes the ratio meaningless
    h[thin], hp[thin] = 0.0, 0.0
    surv_all = np.cumprod(1 - h - hp)
    s_prev = np.concatenate([[1.0], surv_all[:-1]])
    cif = np.cumsum(s_prev * h)
    km = 1 - np.cumprod(1 - h)
    months = np.arange(max_m + 1)
    return {"months": months, "at_risk": at_risk, "defaults": d, "prepaid": pre, "hazard": h,
            "prepay_hazard": hp, "cif": cif, "km": km}


# --------------------------------------------------------------------------------------
# LGD / EAD
# --------------------------------------------------------------------------------------
def amortised_balance(P, r_annual_pct, A, m):
    """Level-payment balance after m payments (clipped at 0)."""
    r = np.asarray(r_annual_pct, float) / 1200.0
    P, A, m = np.asarray(P, float), np.asarray(A, float), np.asarray(m, float)
    g = (1 + r) ** m
    with np.errstate(divide="ignore", invalid="ignore"):
        b = np.where(r > 0, P * g - A * (g - 1) / r, P - A * m)
    return np.clip(b, 0, None)


def lgd_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Charged-off loans with realised EAD, realised recovery and LGD (workout, undiscounted).

    EAD  = funded principal - principal repaid before charge-off (what was owed at default).
    rr   = (recoveries - collection fees) / EAD, clipped to [0,1];   LGD = 1 - rr.
    The charge-off date is not published: est. date = issue + months to last payment + 4m
    (LendingClub charges off at 120+ days past due).
    """
    co = df[df["loan_status"].astype(str).isin(CHARGED_OFF_STATUS)].copy()
    co["ead_obs"] = co["funded_amnt"] - co["total_rec_prncp"]
    co = co[co["ead_obs"] > 1.0].copy()
    mt = co["mtlp"].fillna(0).astype(float)
    co["mtlp_f"] = mt
    co["co_date"] = co["issue_d"] + pd.to_timedelta((mt + CO_LAG_M) * 30.4375, unit="D")
    co["co_year"] = co["co_date"].dt.year
    net_rec = co["recoveries"].fillna(0) - co["collection_recovery_fee"].fillna(0)
    co["rec_rate"] = (net_rec / co["ead_obs"]).clip(0, 1)
    co["lgd_obs"] = 1.0 - co["rec_rate"]
    co["any_rec"] = (net_rec > 0).astype(int)
    # recovery-lag-discounted LGD (assumption: cash arrives ~12 months after charge-off, EIR discount)
    disc = (1 + co["int_rate"] / 100.0) ** (-12 / 12.0)
    co["lgd_disc"] = 1.0 - (co["rec_rate"] * disc)
    return co


LGD_NUM = ["loan_amnt", "funded_amnt", "term_months", "int_rate", "installment", "grade_n",
           "emp_length_years", "annual_inc", "dti", "fico_mid", "revol_util", "open_acc",
           "total_acc", "mort_acc", "mtlp_f", "ead_obs", "ead_ratio"]
LGD_CAT = ["purpose", "home_ownership", "verification_status", "addr_state"]


def lgd_features(co: pd.DataFrame) -> pd.DataFrame:
    f = co.copy()
    f["grade_n"] = f["grade"].astype(str).map({g: i for i, g in enumerate("ABCDEFG", 1)}).astype(float)
    f["ead_ratio"] = f["ead_obs"] / f["funded_amnt"].clip(lower=1)
    return f[LGD_NUM + LGD_CAT]


def lgd_metrics(y, p, base) -> dict:
    y, p = np.asarray(y, float), np.asarray(p, float)
    rmse = float(np.sqrt(np.mean((y - p) ** 2)))
    mae = float(np.mean(np.abs(y - p)))
    rmse_b = float(np.sqrt(np.mean((y - base) ** 2)))
    r2 = 1 - np.sum((y - p) ** 2) / np.sum((y - base) ** 2)   # skill vs the training-mean benchmark
    rho = float(stats.spearmanr(y, p).correlation) if np.std(p) > 0 else 0.0
    return {"n": int(len(y)), "mean_actual": round(float(y.mean()), 5), "mean_pred": round(float(p.mean()), 5),
            "rmse": round(rmse, 5), "mae": round(mae, 5), "rmse_mean_baseline": round(rmse_b, 5),
            "skill_vs_mean": round(float(r2), 5), "spearman": round(rho, 5)}


def bootstrap_rmse_gain(y, p, base, n_iter=1000, seed=SEED) -> dict:
    """Paired bootstrap of RMSE(mean baseline) - RMSE(model); > 0 means the model beats the mean."""
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y, float), np.asarray(p, float)
    se_m, se_b = (y - p) ** 2, (y - base) ** 2
    n = len(y)
    d = np.empty(n_iter)
    for i in range(n_iter):
        s = rng.integers(0, n, n)
        d[i] = math.sqrt(se_b[s].mean()) - math.sqrt(se_m[s].mean())
    return {"gain": round(float(math.sqrt(se_b.mean()) - math.sqrt(se_m.mean())), 6),
            "ci95": [round(float(np.percentile(d, 2.5)), 6), round(float(np.percentile(d, 97.5)), 6)],
            "p_model_not_better": round(float((d <= 0).mean()), 4)}


# --------------------------------------------------------------------------------------
# staging + ECL engine (contractual life, EIR discounting, per-loan hazards)
# --------------------------------------------------------------------------------------
def project_ecl(term, mob, bal, int_rate, installment, pd12, stage, lgd, haz36, haz60,
                pool_p12, horizon12=12, pd_mult=1.0, max_h=60, pre36=None, pre60=None):
    """Marginal-PD x LGD x EAD x DF summed month by month.

    Per-loan hazard at seasoning s:  h_i(s) = 1 - (1 - h_term(s)) ** m_i , with the
    relative-risk multiplier  m_i = ln(1 - PD12_i) / ln(1 - pool_P12(term)).
    Stage 1 sums the next 12 months; Stage 2 sums the remaining contractual life;
    Stage 3 is credit-impaired: ECL = LGD x balance (PD = 1).  EAD_k = scheduled balance
    after k payments; DF_k = (1 + EIR)^(-k/12).  If `pre36/pre60` (empirical monthly
    prepayment hazards by month on book) are given, survival also falls with prepayment, so
    the expected life shortens; without them the projection assumes nobody prepays
    (conservative).  `pool_p12` should be the pooled 12-month default probability on the same
    basis (cumulative incidence when prepayment is modelled).
    """
    n = len(term)
    term, mob = np.asarray(term, int), np.asarray(mob, int)
    remaining = np.clip(term - mob, 1, max_h)
    pd12 = np.clip(np.asarray(pd12, float) * pd_mult, 1e-5, 0.95)
    pool = np.where(term == 36, pool_p12[36], pool_p12[60])
    m = np.log1p(-pd12) / np.log1p(-pool)
    r = np.asarray(int_rate, float) / 1200.0
    eir = np.asarray(int_rate, float) / 100.0
    bal0 = np.asarray(bal, float)
    A = np.asarray(installment, float)
    surv = np.ones(n)
    b = bal0.copy()
    ecl = np.zeros(n)
    pd_h = np.zeros(n)            # PD over the stage horizon (12m for S1, lifetime for S2)
    horizon = np.where(stage == 1, np.minimum(horizon12, remaining), remaining)
    kmax = int(horizon.max()) if n else 0
    lgd_arr = np.broadcast_to(np.asarray(lgd, float), (n,))
    for k in range(1, kmax + 1):
        s = np.clip(mob + k, 0, len(haz36) - 1)
        hz = np.where(term == 36, haz36[s], haz60[s])
        h = 1.0 - (1.0 - np.clip(hz, 0, 0.999)) ** m
        hp = (np.where(term == 36, pre36[s], pre60[s]) if pre36 is not None else 0.0)
        b = np.maximum(b * (1 + r) - A, 0.0)
        active = (k <= horizon) & (stage != 3)
        marg = surv * h
        df_k = (1 + eir) ** (-k / 12.0)
        ecl += np.where(active, marg * lgd_arr * b * df_k, 0.0)
        pd_h += np.where(active, marg, 0.0)
        surv = surv * (1 - h) * (1 - hp)
    s3 = stage == 3
    ecl = np.where(s3, lgd_arr * bal0, ecl)
    pd_h = np.where(s3, 1.0, pd_h)
    return ecl, pd_h


def sicr_flags(pd_orig, pd_now, status, ratio=2.0, abs_pp=0.01, backstop=True):
    """SICR = relative PD deterioration (>= ratio x AND >= abs_pp) OR 30-DPD backstop."""
    rel = (pd_now >= ratio * pd_orig) & ((pd_now - pd_orig) >= abs_pp)
    back = np.isin(status, ["Late (31-120 days)"]) if backstop else np.zeros(len(pd_now), bool)
    return rel | back, rel, back

Writing rcl_lib.py


In [2]:
import os, sys, glob, json, time, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
import rcl_lib as L
from IPython.display import display

ON_KAGGLE = os.path.exists("/kaggle/input")
FAST = os.environ.get("RCL_FAST", "0") == "1"              # local smoke mode only
SAMPLE_FRAC = float(os.environ.get("RCL_SAMPLE", "0.08")) if FAST else None
ART = Path("/kaggle/working/artifacts") if ON_KAGGLE else Path(os.environ.get("RCL_ART", "artifacts"))
for sub in ("tables", "figures", "models"):
    (ART / sub).mkdir(parents=True, exist_ok=True)
M = {"run": {"fast": FAST, "on_kaggle": ON_KAGGLE, "sample_frac": SAMPLE_FRAC}}
T_START = time.time()

def tick(msg): L.log(msg)
def save_table(df, name, index=False): df.to_csv(ART / "tables" / name, index=index)
def save_fig(fig, name):
    fig.savefig(ART / "figures" / name, dpi=120, bbox_inches="tight"); plt.close(fig)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
print("kaggle:", ON_KAGGLE, "| FAST:", FAST, "| artifacts:", ART.resolve())

kaggle: True | FAST: False | artifacts: /kaggle/working/artifacts


## Cell 1 - Load the book and audit the file

**Banker's note.** Source is the public LendingClub 2007-2018Q4 accepted-loan file (151 columns, we read ~40). The file ends with summary footer rows that must be dropped. The data are a point-in-time snapshot: every age, 'months on book' and censoring decision is measured against that snapshot, so we read the snapshot month from the data (latest last-payment month) instead of assuming it.

**Plain-English note.** We load the loan file and check the basics: how many rows, are there junk rows at the bottom, and as-of which month the file describes. Getting the as-of month wrong makes every loan look three months younger than it is.

In [3]:
def find_data():
    env = os.environ.get("RCL_DATA")
    if env and os.path.isfile(env): return env
    hits = [p for p in glob.glob("/kaggle/input/**/*accepted*2018*", recursive=True) if os.path.isfile(p)]
    hits = sorted(hits, key=lambda p: (not p.endswith(".gz"), p))
    if not hits: raise FileNotFoundError("accepted_2007_to_2018Q4 file not found; set RCL_DATA for local runs")
    return hits[0]

CACHE = os.environ.get("RCL_CACHE")
if CACHE and os.path.exists(CACHE):
    raw = pd.read_parquet(CACHE)
    ap = CACHE.replace(".parquet", ".audit.json")
    load_audit = json.load(open(ap)) if os.path.exists(ap) else {"raw_rows": None, "footer_rows_dropped": None}
    DATA = "(local parquet cache) " + CACHE
else:
    DATA = find_data()
    raw, load_audit = L.load_accepted(DATA, sample_frac=SAMPLE_FRAC)
tick(f"loaded {len(raw):,} loan rows from {DATA}")
df = L.derive_basics(raw)
SNAP_IDX = df.attrs["snap_idx"]
snap_label = f"{SNAP_IDX // 12}-{SNAP_IDX % 12 or 12:02d}"
bad_age = float(((df["mtlp"] > df["mob"]).sum()) / df["mtlp"].notna().sum())
audit_rows = {
    "source_file": os.path.basename(str(DATA)),
    "raw_rows": load_audit.get("raw_rows"), "footer_rows_dropped": load_audit.get("footer_rows_dropped"),
    "rows_kept": int(len(df)), "snapshot_month": snap_label,
    "share_last_payment_after_snapshot_age": round(bad_age, 6),
    "issue_range": [str(df.issue_d.min().date()), str(df.issue_d.max().date())],
}
M["data"] = audit_rows
display(pd.Series(audit_rows).to_frame("value"))
assert bad_age < 0.001, "months-to-last-payment exceeds loan age: snapshot month is wrong"
print("loan_status counts:"); display(df["loan_status"].value_counts().to_frame("n"))

[     86s] loaded 2,260,668 loan rows from /kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz


,value
source_file,accepted_2007_to_2018Q4.csv.gz
raw_rows,2260701
footer_rows_dropped,33
rows_kept,2260668
snapshot_month,2019-03
share_last_payment_after_snapshot_age,0.0
issue_range,"[2007-06-01, 2018-12-01]"


loan_status counts:


,n
loan_status,
Fully Paid,1076751
Current,878317
Charged Off,268559
Late (31-120 days),21467
In Grace Period,8436
Late (16-30 days),4349
Does not meet the credit policy. Status:Fully Paid,1988
Does not meet the credit policy. Status:Charged Off,761
Default,40


## Cell 2 - Framing: default definition, window, duplicates, labels, leakage audit

**Banker's note.** Default = 90+ days past due inside the first 12 months on book. LendingClub publishes no delinquency history, so we observe it through the final status and the month of the last payment: a loan that ended Charged Off / Default / Late(31-120) and whose last payment was in month <= 9 had missed three payments by month 12. Loans never paid at all (no last-payment date) are first-payment defaults and are bad. Everything else with enough age is good, **including loans still Current** - they survived the window. Age >= 15 months guarantees the 12-month window plus the ~3 months to reach 90 DPD and the charge-off posting lag have been observed. The legacy label kept only Fully Paid as good, which discarded every surviving Current loan and so inflated the bad rate for recent vintages; the legacy rule is shown for comparison only. Duplicates are removed BEFORE any split. Post-origination fields and LendingClub's own grade/rate/instalment (its own risk output) are never features.

**Plain-English note.** We decide exactly what 'a bad loan' means, make sure no loan appears twice, and list the columns we refuse to use because they only exist after the loan has already gone wrong (using them would be cheating).

In [4]:
df, dup_audit = L.remove_duplicates(df)
M["data"]["duplicates"] = dup_audit
display(pd.Series(dup_audit).to_frame("value"))
df = L.add_labels(df)
df = L.add_split(df)
assert (df["split"] != "unassigned").all()

lab = df[df.resolved_12m]
M["label"] = {
    "definition": "bad_12m=1: ended Charged Off/Default/Late(31-120) and last payment in month<=9 (90+ DPD within 12m); first-payment defaults included; "
                  "0: all other loans aged >=15 months incl. Current; undefined: age<15m or active loan that stopped paying inside the window",
    "min_age_months": L.MIN_AGE, "last_payment_max_month": L.LAST_PAY_MAX,
    "n_first_payment_defaults": int(df["never_paid"].sum()),
    "n_indeterminate_excluded": int(((df.mob >= L.MIN_AGE) & ~df.resolved_12m).sum()),
    "n_labelled": int(len(lab)), "bad_rate_labelled": round(float(lab.bad_12m.mean()), 5),
}
comp = pd.crosstab(lab.loan_status.astype(str), lab.bad_12m.astype(int)); save_table(comp.reset_index(), "label_status_composition.csv"); display(comp)
M["label"]["bad_by_final_status"] = {k: int(v) for k, v in comp[1].items()} if 1 in comp else {}
rows = []
for s in ["train", "valid", "oot", "psi_only"]:
    d = df[df.split == s]
    new = d[d.resolved_12m]; old = d[d.bad_12m_legacy.notna()]
    rows.append({"split": s, "loans": len(d),
                 "labelled_new": len(new), "bad_rate_new": new.bad_12m.mean() if len(new) else np.nan,
                 "labelled_legacy": len(old), "bad_rate_legacy": old.bad_12m_legacy.mean() if len(old) else np.nan,
                 "share_Current_in_new_pop": float((new.loan_status.astype(str) == "Current").mean()) if len(new) else np.nan})
lab_cmp = pd.DataFrame(rows)
save_table(lab_cmp, "label_new_vs_legacy_by_split.csv")
M["label"]["new_vs_legacy_by_split"] = lab_cmp.round(5).to_dict("records")
display(lab_cmp.round(4))

# bad rate by issue year, both definitions
by_year = df.groupby(df.issue_d.dt.year).apply(lambda d: pd.Series({
    "new": d.bad_12m.mean(), "legacy": d.bad_12m_legacy.mean(), "n_new": d.resolved_12m.sum()}))
fig, ax = plt.subplots(figsize=(8, 3.6))
by_year[["new", "legacy"]].plot(ax=ax, marker="o")
ax.set_title("12-month bad rate by issue year: corrected label vs legacy label"); ax.set_ylabel("bad rate"); ax.set_xlabel("issue year")
save_fig(fig, "bad_rate_by_vintage_new_vs_legacy.png"); display(by_year.round(4))

# leakage audit
LEAK = pd.DataFrame([
    {"field": f, "reason": r} for f, r in [
        ("out_prncp / total_rec_prncp / total_rec_int / total_pymnt", "post-origination cash flows"),
        ("recoveries / collection_recovery_fee", "post-charge-off outcome (used only for LGD, never PD)"),
        ("last_pymnt_d / last_pymnt_amnt / last_credit_pull_d", "post-origination servicing (last_pymnt_d defines the label)"),
        ("last_fico_range_high/low", "refreshed bureau score pulled after origination (used only for staging at the reporting date)"),
        ("loan_status / settlement_* / debt_settlement_flag", "the outcome itself"),
        ("grade / sub_grade / int_rate / installment", "LendingClub's own risk-based price: a prior model's PD output")]])
save_table(LEAK, "leakage_audit_dropped_fields.csv")
display(LEAK)
assert not (set(L.PD_FEATURES) & set(L.LEAKAGE_FIELDS)), "leakage field in PD features"
# automated check: no single application feature should rank the label almost perfectly
from sklearn.metrics import roc_auc_score
tr = df[(df.split == "train") & df.resolved_12m].sample(200_000, random_state=42) if (df.split == "train").sum() > 250_000 else df[(df.split == "train") & df.resolved_12m]
chk = []
for c in L.NUMERIC:
    v = pd.to_numeric(tr[c], errors="coerce"); ok = v.notna()
    a = roc_auc_score(tr.bad_12m[ok], v[ok]); chk.append({"feature": c, "single_feature_auc": max(a, 1 - a)})
chk = pd.DataFrame(chk).sort_values("single_feature_auc", ascending=False)
save_table(chk, "single_feature_auc_leak_check.csv"); display(chk.head(6).round(3))
assert chk.single_feature_auc.max() < 0.75, "a single feature is suspiciously predictive: check for leakage"

,value
rows_in,2260668
duplicate_ids_removed,0
duplicate_applications_removed,0
rows_out,2260668


bad_12m,0,1
loan_status,,
Charged Off,186989,72703
Current,451136,0
Default,33,0
Does not meet the credit policy. Status:Charged Off,507,254
Does not meet the credit policy. Status:Fully Paid,1988,0
Fully Paid,1029307,0
In Grace Period,5517,0
Late (16-30 days),2746,0
Late (31-120 days),14246,0


,split,loans,labelled_new,bad_rate_new,labelled_legacy,bad_rate_legacy,share_Current_in_new_pop
0,train,887440,887440,0.0372,723182,0.0676,0.0622
1,valid,434407,434407,0.0470,254268,0.1157,0.3086
2,oot,443579,443579,0.0440,157414,0.1732,0.5903
3,psi_only,495242,0,NaN,26457,0.3142,NaN


,new,legacy,n_new
issue_d,,,
2007,0.0514,0.0507,603.0
2008,0.0656,0.0647,2393.0
2009,0.0464,0.0574,5281.0
2010,0.0345,0.0463,12537.0
2011,0.0355,0.0545,21721.0
2012,0.0373,0.0611,53367.0
2013,0.0318,0.0529,134814.0
2014,0.0344,0.0631,235629.0
2015,0.0404,0.0783,421095.0


,field,reason
0,out_prncp / total_rec_prncp / total_rec_int / ...,post-origination cash flows
1,recoveries / collection_recovery_fee,"post-charge-off outcome (used only for LGD, ne..."
2,last_pymnt_d / last_pymnt_amnt / last_credit_p...,post-origination servicing (last_pymnt_d defin...
3,last_fico_range_high/low,refreshed bureau score pulled after originatio...
4,loan_status / settlement_* / debt_settlement_flag,the outcome itself
5,grade / sub_grade / int_rate / installment,LendingClub's own risk-based price: a prior mo...


,feature,single_feature_auc
13,fico_mid,0.581
6,inq_last_6mths,0.568
4,dti,0.565
3,annual_inc,0.560
12,mort_acc,0.546
1,term_months,0.545


## Cell 3 - Out-of-time split and data checks

**Banker's note.** Time split, never random: train = issued up to 2015, valid = 2016 (model selection and calibration), oot = 2017 (every headline number), 2018 vintages are too young to carry a 12-month label and are used only for stability (PSI) and the live book. Missing values, ranges and sentinel codes are checked on the training window only.

**Plain-English note.** We cut the data by calendar: learn from the older loans, choose and tune on 2016, and take one final exam on 2017 loans that nobody touched until the end.

In [5]:
pop = df[df.resolved_12m].copy()
parts = {s: pop[pop.split == s] for s in ["train", "valid", "oot"]}
X = {s: p[L.PD_FEATURES].astype({c: "object" for c in L.CATEGORICAL}) for s, p in parts.items()}
Y = {s: p.bad_12m.to_numpy(float) for s, p in parts.items()}
split_tbl = pd.DataFrame([{"split": s, "n": len(Y[s]), "bad_rate": Y[s].mean(), "first_issue": parts[s].issue_d.min().date(),
                           "last_issue": parts[s].issue_d.max().date()} for s in parts])
save_table(split_tbl, "split_summary.csv"); display(split_tbl.round(4))
M["split"] = split_tbl.assign(first_issue=split_tbl.first_issue.astype(str), last_issue=split_tbl.last_issue.astype(str)).round(5).to_dict("records")
assert parts["train"].issue_d.max() < parts["valid"].issue_d.min() < parts["oot"].issue_d.min()
assert not (set(parts["train"].id) & set(parts["oot"].id)) and not (set(parts["train"].id) & set(parts["valid"].id))

tr = parts["train"]
miss = pd.DataFrame({"missing_share": tr[L.PD_FEATURES].isna().mean(),
                     "min": [pd.to_numeric(tr[c], errors="coerce").min() if c in L.NUMERIC else np.nan for c in L.PD_FEATURES],
                     "p50": [pd.to_numeric(tr[c], errors="coerce").median() if c in L.NUMERIC else np.nan for c in L.PD_FEATURES],
                     "p99.9": [pd.to_numeric(tr[c], errors="coerce").quantile(0.999) if c in L.NUMERIC else np.nan for c in L.PD_FEATURES],
                     "max": [pd.to_numeric(tr[c], errors="coerce").max() if c in L.NUMERIC else np.nan for c in L.PD_FEATURES]})
save_table(miss.reset_index().rename(columns={"index": "feature"}), "data_quality_features.csv")
display(miss.round(3))
print("Treatment: numerics median-imputed inside every model pipeline; extreme values are handled by quantile binning (WoE) or tree splits; "
      "no row is dropped for outliers. dti < 0 or > 100 and revol_util > 150 are kept as reported (rare).")

,split,n,bad_rate,first_issue,last_issue
0,train,887440,0.0372,2007-06-01,2015-12-01
1,valid,434407,0.0470,2016-01-01,2016-12-01
2,oot,443579,0.0440,2017-01-01,2017-12-01


,missing_share,min,p50,p99.9,max
loan_amnt,0.000,500.0,13000.00,35000.000,35000.0
term_months,0.000,36.0,36.00,60.000,60.0
emp_length_years,0.051,0.0,6.00,10.000,10.0
annual_inc,0.000,0.0,65000.00,550000.000,9500000.0
dti,0.000,0.0,17.66,39.720,999.0
delinq_2yrs,0.000,0.0,0.00,8.000,39.0
inq_last_6mths,0.000,0.0,0.00,6.000,33.0
open_acc,0.000,0.0,11.00,39.000,90.0
pub_rec,0.000,0.0,0.00,5.000,86.0
revol_bal,0.000,0.0,11875.00,260433.202,2904836.0


Treatment: numerics median-imputed inside every model pipeline; extreme values are handled by quantile binning (WoE) or tree splits; no row is dropped for outliers. dti < 0 or > 100 and revol_util > 150 are kept as reported (rare).


## Cell 4 - Univariate analysis: weight of evidence and information value

**Banker's note.** WoE/IV on the training window only, quantile bins (10) for numerics. IV < 0.02 is uninformative, 0.1-0.3 medium, > 0.5 is a leakage suspect. WoE bins also drive the classical scorecard in the ladder below.

**Plain-English note.** For each applicant detail we ask: do the bins of this detail separate people who stop paying from people who do not? IV is one number for how much it helps.

In [6]:
samp = parts["train"].sample(min(300_000, len(parts["train"])), random_state=42)
xs = samp[L.PD_FEATURES].astype({c: "object" for c in L.CATEGORICAL})
iv, wbins = L.iv_table(xs, samp.bad_12m.to_numpy(float), n_bins=10)
save_table(iv, "iv_table.csv"); save_table(wbins, "woe_bins.csv"); display(iv.round(4))
M["univariate"] = {"iv_top5": iv.head(5).round(4).to_dict("records"), "n_iv_below_0.02": int((iv.iv < 0.02).sum()), "n_iv_above_0.5": int((iv.iv > 0.5).sum())}
fig, ax = plt.subplots(figsize=(7, 4.5)); iv.iloc[::-1].plot.barh(x="feature", y="iv", ax=ax, legend=False, color="#4C72B0")
ax.axvline(0.02, color="grey", ls=":"); ax.axvline(0.1, color="grey", ls="--"); ax.set_title("Information value by feature (train)"); save_fig(fig, "iv_bar.png")
top = iv.feature.iloc[0]
b = wbins[wbins.feature == top]
fig, ax = plt.subplots(figsize=(7, 3.5)); ax.bar(range(len(b)), b.bad_rate, color="#C44E52"); ax.set_xticks(range(len(b))); ax.set_xticklabels(b.bin, rotation=60, ha="right", fontsize=7)
ax.set_title(f"Bad rate by bin: {top} (monotone is what a scorecard wants)"); save_fig(fig, "woe_bins_top_feature.png")

,feature,iv,strength
0,fico_mid,0.0907,weak
1,inq_last_6mths,0.0692,weak
2,dti,0.0548,weak
3,purpose,0.0543,weak
4,annual_inc,0.0464,weak
5,verification_status,0.0392,weak
6,term_months,0.0331,weak
7,home_ownership,0.0328,weak
8,mort_acc,0.0313,weak
9,emp_length_years,0.0190,useless (<0.02)


## Cell 5 - Multicollinearity and coefficient-sign check

**Banker's note.** Pairwise |r| > 0.7, then VIF with iterative drop above 5 (reported, not applied to the tree models which tolerate collinearity). A statsmodels Logit on WoE inputs checks convergence, p-values and that every coefficient has the expected sign (WoE here is log(bad/good), so a positive coefficient is the expected direction).

**Plain-English note.** Some details say almost the same thing (open accounts vs total accounts). We measure the overlap and check that the model's directions make business sense.

In [7]:
num = xs[L.NUMERIC].apply(pd.to_numeric, errors="coerce")
corr = num.corr()
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:] if abs(corr.loc[a, b]) > 0.7]
pairs = pd.DataFrame(pairs, columns=["feature_a", "feature_b", "corr"]); save_table(pairs, "high_correlation_pairs.csv"); display(pairs.round(3))
vif, vif_drop = L.vif_table(num)
save_table(vif, "vif_table.csv"); display(vif.round(2)); print("iterative VIF>5 drops:", vif_drop)
M["multicollinearity"] = {"pairs_above_0.7": pairs.round(3).to_dict("records"), "vif_dropped": vif_drop, "max_vif_initial": round(float(vif.vif_initial.max()), 2)}
fig, ax = plt.subplots(figsize=(6.5, 5)); im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7); ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
ax.grid(False); fig.colorbar(im, ax=ax); ax.set_title("Correlation of numeric features (train sample)"); save_fig(fig, "correlation_heatmap.png")

,feature_a,feature_b,corr


,feature,vif_initial,dropped_in_iteration
0,total_acc,2.37,False
1,open_acc,2.23,False
2,revol_util,1.59,False
3,loan_amnt,1.54,False
4,fico_mid,1.51,False
5,annual_inc,1.38,False
6,revol_bal,1.37,False
7,mort_acc,1.33,False
8,dti,1.29,False
9,term_months,1.23,False


iterative VIF>5 drops: []


## Cell 6 - Model ladder under ONE cross-validation scheme

**Banker's note.** Baselines (prior-only dummy; plain untuned logistic), then tuned L2 logistic, WoE scorecard, WoE L1, random forest, LightGBM, XGBoost and CatBoost. Every tunable family is searched with the same StratifiedKFold(3) roc_auc scheme on a seeded 25% subsample of train and then refit on the full train window. WoE is fitted inside each fold. Nothing in this cell sees validation or out-of-time data.

**Plain-English note.** We train a staircase of models from simple to sophisticated, giving each the same fair tuning budget, then see whether the sophistication pays.

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
models, tuning = {}, {}
models["dummy prior"] = Pipeline([("prep", L.one_hot_prep()), ("clf", DummyClassifier(strategy="prior"))]).fit(X["train"], Y["train"])
models["plain logistic (one-hot, untuned C=1)"] = Pipeline([("prep", L.one_hot_prep()), ("clf", LogisticRegression(C=1.0, max_iter=3000, random_state=L.SEED))]).fit(X["train"], Y["train"])
rng = np.random.default_rng(L.SEED)
frac = 0.10 if FAST else 0.25
idx = rng.choice(len(Y["train"]), size=max(2000, int(len(Y["train"]) * frac)), replace=False)
x_search, y_search = X["train"].iloc[idx].reset_index(drop=True), Y["train"][idx]
cv = StratifiedKFold(n_splits=2 if FAST else 3, shuffle=True, random_state=L.SEED)
tick(f"CV subsample n={len(y_search):,}, bad={y_search.mean():.4f}, folds={cv.get_n_splits()}")
families = {}
for cand in L.build_candidates(fast=FAST):
    tick(f"tuning {cand['name']}")
    models[cand["name"]], tuning[cand["name"]] = L.search_and_refit(cand, x_search, y_search, cv, X["train"], Y["train"])
    families[cand["name"]] = cand["family"]
M["tuning"] = tuning
write = pd.DataFrame([{"model": k, **{kk: vv for kk, vv in v.items() if kk != "best_params"}, "best_params": json.dumps(v["best_params"])} for k, v in tuning.items()])
save_table(write, "tuning_report.csv"); display(write)

[    171s] CV subsample n=221,860, bad=0.0365, folds=3
[    175s] tuning LogisticRegression L2 (one-hot, tuned C)
[    280s]   LogisticRegression L2 (one-hot, tuned C): CV auc=0.6625 (4x3 fits, 67s) {'clf__C': 1.0}
[    280s] tuning WOE logistic scorecard (tuned C x bins)
[    322s]   WOE logistic scorecard (tuned C x bins): CV auc=0.6621 (8x3 fits, 36s) {'clf__C': 0.1, 'woe__n_bins': 10}
[    322s] tuning WOE L1 logistic (lasso, tuned C)
[    344s]   WOE L1 logistic (lasso, tuned C): CV auc=0.6616 (4x3 fits, 16s) {'clf__C': 1.0}
[    344s] tuning RandomForest (tuned)
[   1179s]   RandomForest (tuned): CV auc=0.6706 (6x3 fits, 600s) {'clf__n_estimators': 250, 'clf__min_samples_leaf': 150, 'clf__max_features': 'sqrt', 'clf__max_depth': 18, 'clf__class_weight': 'balanced_subsample'}
[   1179s] tuning LightGBM (tuned)
[   1348s]   LightGBM (tuned): CV auc=0.6744 (15x3 fits, 155s) {'clf__subsample': 0.7, 'clf__reg_lambda': 5.0, 'clf__reg_alpha': 0.0, 'clf__num_leaves': 31, 'clf__n_estimato

,model,search,n_candidates,n_folds,best_cv_roc_auc,best_cv_gini,search_seconds,refit_seconds,best_params
0,"LogisticRegression L2 (one-hot, tuned C)",GridSearchCV,4,3,0.66251,0.32503,67.5,36.8,"{""clf__C"": 1.0}"
1,WOE logistic scorecard (tuned C x bins),GridSearchCV,8,3,0.66214,0.32427,35.9,6.4,"{""clf__C"": 0.1, ""woe__n_bins"": 10}"
2,"WOE L1 logistic (lasso, tuned C)",GridSearchCV,4,3,0.66160,0.32320,15.8,6.6,"{""clf__C"": 1.0}"
3,RandomForest (tuned),RandomizedSearchCV,6,3,0.67056,0.34113,600.1,234.3,"{""clf__n_estimators"": 250, ""clf__min_samples_l..."
4,LightGBM (tuned),RandomizedSearchCV,15,3,0.67443,0.34886,155.3,13.9,"{""clf__subsample"": 0.7, ""clf__reg_lambda"": 5.0..."
5,XGBoost (tuned),RandomizedSearchCV,12,3,0.67566,0.35132,121.9,11.1,"{""clf__subsample"": 0.9, ""clf__reg_lambda"": 5.0..."
6,CatBoost (tuned),RandomizedSearchCV,10,3,0.67750,0.35499,262.2,51.3,"{""clf__random_strength"": 1.0, ""clf__learning_r..."


## Cell 7 - Champion rule FROZEN before any out-of-time scoring

**Banker's note.** Rule (written to `champion_rule.json` before the 2017 data is scored by any model): rank the candidates by Gini on the 2016 validation window; the highest wins; if the winner leads the WoE scorecard by less than 0.002 Gini, the more interpretable WoE scorecard is chosen instead. The dummy and the untuned logistic are floors, not candidates.

**Plain-English note.** We write the rule for picking the winner on paper first, using only the 2016 exam. Only after that do we open the sealed 2017 envelope, so we cannot pick whichever model happens to look best on it.

In [9]:
valid_tbl = []
for name, m in models.items():
    pv = m.predict_proba(X["valid"])[:, 1]; pt = m.predict_proba(X["train"])[:, 1]
    valid_tbl.append({"model": name, "family": families.get(name, "baseline"), "valid_gini": L.gini(Y["valid"], pv), "train_gini": L.gini(Y["train"], pt)})
valid_tbl = pd.DataFrame(valid_tbl).sort_values("valid_gini", ascending=False).reset_index(drop=True)
cands = valid_tbl[~valid_tbl.model.isin(["dummy prior", "plain logistic (one-hot, untuned C=1)"])]
WOE_NAME = "WOE logistic scorecard (tuned C x bins)"
best = cands.iloc[0]
woe_valid = float(cands.loc[cands.model == WOE_NAME, "valid_gini"].iloc[0])
champion_name = WOE_NAME if (best.valid_gini - woe_valid) < 0.002 else best.model
challenger_name = [m for m in cands.model if m not in (champion_name, WOE_NAME)][0] if champion_name != WOE_NAME else cands.model.iloc[0]
rule = {"rule": "highest validation-window Gini; a lead of < 0.002 over the WoE scorecard goes to the WoE scorecard; floors are not candidates",
        "frozen_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "oot_outcomes_used_to_write_rule": False,
        "candidates_valid_gini": {r.model: round(r.valid_gini, 5) for r in cands.itertuples()},
        "champion": champion_name, "runner_up": challenger_name, "classical_benchmark": WOE_NAME}
L.write_json(rule, ART / "champion_rule.json")
M["champion_rule"] = rule
save_table(valid_tbl, "validation_roster.csv"); display(valid_tbl.round(4))
print("CHAMPION (frozen):", champion_name, "| classical benchmark:", WOE_NAME, "| runner-up:", challenger_name)

,model,family,valid_gini,train_gini
0,CatBoost (tuned),tree ensemble,0.3570,0.4050
1,XGBoost (tuned),tree ensemble,0.3525,0.4179
2,LightGBM (tuned),tree ensemble,0.3479,0.4059
3,RandomForest (tuned),tree ensemble,0.3381,0.6156
4,"LogisticRegression L2 (one-hot, tuned C)",classical linear,0.3178,0.3392
5,"plain logistic (one-hot, untuned C=1)",baseline,0.3178,0.3392
6,WOE logistic scorecard (tuned C x bins),credit scorecard,0.3141,0.3435
7,"WOE L1 logistic (lasso, tuned C)",classical linear,0.3113,0.3423
8,dummy prior,baseline,0.0000,0.0000


CHAMPION (frozen): CatBoost (tuned) | classical benchmark: WOE logistic scorecard (tuned C x bins) | runner-up: XGBoost (tuned)


## Cell 8 - Out-of-time scoring and significance (champion vs WoE scorecard)

**Banker's note.** Now the 2017 window is scored for every model. The champion is compared with the **WoE scorecard** (the incumbent-style benchmark), the L2 logistic, and the runner-up, using a paired DeLong test and a paired 1,000-draw bootstrap of the Gini difference. Discrimination is compared on **raw model scores**: isotonic calibration creates ties and is a calibration step, not a ranking step.

**Plain-English note.** We take the final exam and ask a fair question: is the fancy model really better than the simple scorecard, or could the gap be luck? The test is run against the scorecard, not against another fancy model.

In [10]:
raw_oot = {n: m.predict_proba(X["oot"])[:, 1] for n, m in models.items()}
raw_valid = {n: m.predict_proba(X["valid"])[:, 1] for n, m in models.items()}
roster = []
for n in models:
    e = L.evaluate(Y["oot"], raw_oot[n]); v = L.evaluate(Y["valid"], raw_valid[n])
    roster.append({"model": n, "family": families.get(n, "baseline"), "valid_gini": v["gini"], "oot_auc": e["auc"], "oot_gini": e["gini"], "oot_ks": e["ks"],
                   "oot_pr_auc": e["pr_auc"], "oot_brier": e["brier"], "oot_lift_10": e["lift_10"]})
roster = pd.DataFrame(roster).sort_values("oot_gini", ascending=False).reset_index(drop=True)
save_table(roster, "model_roster_oot.csv"); display(roster.round(4))
M["pd"] = {"roster": roster.round(5).to_dict("records"), "champion": champion_name, "benchmark": WOE_NAME}

NB = 50 if FAST else 1000
sig = {}
for label, other in [("vs_woe_scorecard", WOE_NAME), ("vs_l2_logistic", "LogisticRegression L2 (one-hot, tuned C)"), ("vs_runner_up", challenger_name)]:
    if other == champion_name: continue
    dl = L.delong_test(Y["oot"], raw_oot[champion_name], raw_oot[other])
    bs = L.bootstrap_gini_delta(Y["oot"], raw_oot[champion_name], raw_oot[other], n_iter=NB)
    sig[label] = {"champion": champion_name, "comparator": other, "delong": dl, "bootstrap": bs}
    tick(f"{label}: dGini={dl['delta_gini']:+.4f} DeLong p={dl['p_value']:.3g} | bootstrap ci95={bs['ci95']} p={bs['p_two_sided']}")
M["pd"]["significance"] = sig
M["pd"]["champion_oot"] = L.evaluate(Y["oot"], raw_oot[champion_name])
M["pd"]["benchmark_oot"] = L.evaluate(Y["oot"], raw_oot[WOE_NAME])
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(5.5, 5))
for n in [champion_name, WOE_NAME, "dummy prior"]:
    f, t, _ = roc_curve(Y["oot"], raw_oot[n]); ax.plot(f, t, label=f"{n.split(' (')[0]} (Gini {L.gini(Y['oot'], raw_oot[n]):.3f})")
ax.plot([0, 1], [0, 1], "k:", lw=0.8); ax.legend(fontsize=7); ax.set_title("Out-of-time ROC (2017 vintages)"); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
save_fig(fig, "roc_oot.png")

,model,family,valid_gini,oot_auc,oot_gini,oot_ks,oot_pr_auc,oot_brier,oot_lift_10
0,CatBoost (tuned),tree ensemble,0.3570,0.6764,0.3528,0.2557,0.0851,0.0414,2.3382
1,XGBoost (tuned),tree ensemble,0.3525,0.6736,0.3472,0.2496,0.0843,0.0414,2.2813
2,LightGBM (tuned),tree ensemble,0.3479,0.6711,0.3423,0.2450,0.0829,0.0414,2.2623
3,RandomForest (tuned),tree ensemble,0.3381,0.6680,0.3361,0.2399,0.0815,0.1743,2.2172
4,WOE logistic scorecard (tuned C x bins),credit scorecard,0.3141,0.6567,0.3134,0.2259,0.0750,0.0418,2.0634
5,"WOE L1 logistic (lasso, tuned C)",classical linear,0.3113,0.6563,0.3125,0.2257,0.0748,0.0418,2.0665
6,"plain logistic (one-hot, untuned C=1)",baseline,0.3178,0.6476,0.2952,0.2153,0.0774,0.0421,2.1829
7,"LogisticRegression L2 (one-hot, tuned C)",classical linear,0.3178,0.6476,0.2952,0.2153,0.0774,0.0421,2.1829
8,dummy prior,baseline,0.0000,0.5000,0.0000,0.0000,0.0440,0.0421,1.0076


[   2181s] vs_woe_scorecard: dGini=+0.0394 DeLong p=2.06e-87 | bootstrap ci95=[0.03551, 0.04352] p=0.0
[   2473s] vs_l2_logistic: dGini=+0.0576 DeLong p=1.24e-88 | bootstrap ci95=[0.05172, 0.06386] p=0.0
[   2760s] vs_runner_up: dGini=+0.0057 DeLong p=3.31e-10 | bootstrap ci95=[0.00384, 0.00748] p=0.0


## Cell 9 - Calibration on VALIDATION only

**Banker's note.** Calibration maps the ranking score to a probability. The method (raw / isotonic / Platt) is chosen by 5-fold cross-validated ECE inside the 2016 validation window; the calibrator is then fitted on all of validation and applied unchanged to 2017. ECE is on deciles. The 2017 outcomes are never used to fit or choose anything.

**Plain-English note.** A model can rank people well and still say '10% chance' when the truth is 6%. Calibration fixes the 'how many' using last year's results, and we then check it on the following year.

In [11]:
p_valid_raw, p_oot_raw = raw_valid[champion_name], raw_oot[champion_name]
cal_cv = {m: L.cv_ece_of_calibrator(m, p_valid_raw, Y["valid"]) for m in ["raw", "isotonic", "platt"]}
cal_sel = min(cal_cv, key=cal_cv.get)
calibs = {m: L.Calibrator(m).fit(p_valid_raw, Y["valid"]) for m in ["raw", "isotonic", "platt"]}
cal_tbl = []
for m, c in calibs.items():
    po = c.predict(p_oot_raw); pv = c.predict(p_valid_raw)
    cal_tbl.append({"method": m, "ece_valid_cv": cal_cv[m], "ece_oot": L.ece_point(Y["oot"], po), "brier_oot": float(np.mean((Y["oot"] - po) ** 2)),
                    "mean_pred_oot": float(po.mean()), "actual_oot": float(Y["oot"].mean()), "gini_oot": L.gini(Y["oot"], po)})
cal_tbl = pd.DataFrame(cal_tbl); save_table(cal_tbl, "calibration_methods.csv"); display(cal_tbl.round(5))
CAL = calibs[cal_sel]
pd_valid, pd_oot = CAL.predict(p_valid_raw), CAL.predict(p_oot_raw)
ece_oot, deciles = L.ece_and_deciles(Y["oot"], pd_oot)
M["calibration"] = {"selected": cal_sel, "selection_rule": "lowest 5-fold CV ECE inside the validation window", "ece_oot_selected": round(ece_oot, 5),
                    "ece_oot_raw": round(L.ece_point(Y["oot"], p_oot_raw), 5), "methods": cal_tbl.round(5).to_dict("records")}
display(pd.DataFrame(deciles)); save_table(pd.DataFrame(deciles), "calibration_deciles_oot.csv")
print("selected calibration:", cal_sel, "| OOT ECE:", round(ece_oot, 5))
fig, ax = plt.subplots(figsize=(5, 5)); d = pd.DataFrame(deciles)
ax.plot(d.expected_pd, d.actual_pd, "o-", label=f"{cal_sel} (ECE {ece_oot:.3f})")
dr = pd.DataFrame(L.ece_and_deciles(Y["oot"], p_oot_raw)[1]); ax.plot(dr.expected_pd, dr.actual_pd, "s--", label="raw", alpha=0.7)
mx = max(d.expected_pd.max(), d.actual_pd.max(), dr.expected_pd.max()) * 1.05; ax.plot([0, mx], [0, mx], "k:"); ax.legend(); ax.set_xlabel("predicted PD (decile mean)"); ax.set_ylabel("actual default rate")
ax.set_title("Reliability on 2017 (out of time)"); save_fig(fig, "reliability_oot.png")

,method,ece_valid_cv,ece_oot,brier_oot,mean_pred_oot,actual_oot,gini_oot
0,raw,0.01069,0.00887,0.04141,0.03512,0.04399,0.35283
1,isotonic,0.00055,0.00154,0.04135,0.04545,0.04399,0.35231
2,platt,0.00066,0.00178,0.04133,0.04547,0.04399,0.35283


,decile,n,expected_pd,actual_pd,difference
0,1,44358,0.00897,0.00922,0.00025
1,2,44358,0.01768,0.01779,0.00011
2,3,44358,0.02418,0.02342,-0.00075
3,4,44358,0.02945,0.02901,-0.00044
4,5,44358,0.03656,0.03652,-0.00004
5,6,44357,0.04299,0.04211,-0.00087
6,7,44358,0.05082,0.04946,-0.00136
7,8,44358,0.05909,0.05830,-0.00079
8,9,44358,0.07261,0.07173,-0.00088
9,10,44358,0.11219,0.10228,-0.00991


selected calibration: isotonic | OOT ECE: 0.00154


## Cell 10 - Backtest: predicted vs actual default rate, and the master scale

**Banker's note.** ECE alone hides level error, so the portfolio-level backtest is reported next to it: mean predicted PD vs realised 12-month default rate on 2016 (calibration window) and 2017 (out of time), with an exact binomial test and Wilson interval. A master scale of up to 8 grades is cut on **validation** quantiles of the calibrated PD and then frozen; each grade is back-tested on 2017 with a binomial test. With ~400k loans the binomial test rejects tiny gaps, so the effect size (ratio actual/predicted) is shown too. A gap larger than +/-10% would be a recalibration trigger.

**Plain-English note.** This is the 'did you forecast the right NUMBER of defaults?' test. We compare the share of loans the model said would default with the share that really did, overall and in each risk grade.

In [12]:
def backtest_row(y, p, name):
    k, n = int(np.sum(y)), len(y)
    r = {"window": name, "n": n, "predicted_pd": float(np.mean(p)), "actual_default_rate": k / n, "ratio_actual_over_pred": (k / n) / float(np.mean(p)),
         "abs_gap_pp": 100 * (k / n - float(np.mean(p)))}
    r.update(L.binom_row(k, n, float(np.mean(p)))); return r
bt = pd.DataFrame([backtest_row(Y["valid"], pd_valid, "valid 2016 (calibration window)"), backtest_row(Y["oot"], pd_oot, "oot 2017 (out of time)")])
bt["ci95_low"] = bt.ci95_actual.apply(lambda c: c[0]); bt["ci95_high"] = bt.ci95_actual.apply(lambda c: c[1]); bt = bt.drop(columns="ci95_actual")
save_table(bt, "backtest_overall.csv"); display(bt.round(5))
M["backtest"] = {"overall": bt.round(6).to_dict("records"), "recalibration_trigger_band": [0.9, 1.1],
                 "oot_within_band": bool(0.9 <= bt.ratio_actual_over_pred.iloc[1] <= 1.1)}

edges = np.unique(np.quantile(pd_valid, np.linspace(0, 1, 9)[1:-1]))
def grade_of(p): return np.searchsorted(edges, p, side="right") + 1
gv, go = grade_of(pd_valid), grade_of(pd_oot)
rows = []
for g in sorted(set(gv) | set(go)):
    for nm, gg, yy, pp in [("valid", gv, Y["valid"], pd_valid), ("oot", go, Y["oot"], pd_oot)]:
        m = gg == g
        if m.sum() == 0: continue
        r = {"grade": int(g), "window": nm, "n": int(m.sum()), "predicted_pd": float(pp[m].mean()), "actual_default_rate": float(yy[m].mean())}
        r["ratio"] = r["actual_default_rate"] / r["predicted_pd"]; r.update({"binom_p": L.binom_row(int(yy[m].sum()), int(m.sum()), r["predicted_pd"])["binom_p"]}); rows.append(r)
grades = pd.DataFrame(rows); save_table(grades, "master_scale_backtest.csv"); display(grades.round(4))
M["backtest"]["master_scale_edges"] = [float(e) for e in edges]; M["backtest"]["grades"] = grades.round(5).to_dict("records")
go_ = grades[grades.window == "oot"]
fig, ax = plt.subplots(figsize=(6.5, 4)); w = 0.38; xg = np.arange(len(go_))
ax.bar(xg - w / 2, go_.predicted_pd, w, label="predicted", color="#4C72B0"); ax.bar(xg + w / 2, go_.actual_default_rate, w, label="actual (2017)", color="#C44E52")
ax.set_xticks(xg); ax.set_xticklabels([f"G{g}" for g in go_.grade]); ax.legend(); ax.set_title("Grade back-test, 2017 out of time"); ax.set_ylabel("12-month default rate"); save_fig(fig, "grade_backtest_oot.png")

# policy simulation: how much of a level gap would a re-level on the first half of 2017 close on the second half?
oot_df = parts["oot"].reset_index(drop=True); h1 = (oot_df.issue_d.dt.month <= 6).to_numpy()
fac = Y["oot"][h1].mean() / pd_oot[h1].mean()
sim = {"relevel_factor_from_2017H1": float(fac), "H2_pred_before": float(pd_oot[~h1].mean()), "H2_pred_after": float(pd_oot[~h1].mean() * fac), "H2_actual": float(Y["oot"][~h1].mean())}
M["backtest"]["recalibration_policy_simulation"] = {**sim, "note": "monitoring-policy illustration only; it is NOT used to produce any headline number"}
display(pd.Series(sim).to_frame("value"))

,window,n,predicted_pd,actual_default_rate,ratio_actual_over_pred,abs_gap_pp,binom_p,ci95_low,ci95_high
0,valid 2016 (calibration window),434407,0.04704,0.04704,1.0000,-0.00000,1.0,0.04642,0.04767
1,oot 2017 (out of time),443579,0.04545,0.04399,0.9677,-0.14681,0.0,0.04339,0.04459


,grade,window,n,predicted_pd,actual_default_rate,ratio,binom_p
0,1,valid,50217,0.0109,0.0109,1.0000,1.0000
1,1,oot,57979,0.0107,0.0106,0.9921,0.8717
2,2,valid,49758,0.0201,0.0201,1.0000,1.0000
3,2,oot,53322,0.0201,0.0203,1.0098,0.7458
4,3,valid,51347,0.0274,0.0274,1.0000,1.0000
5,3,oot,53882,0.0274,0.0268,0.9775,0.3913
6,4,valid,59322,0.0359,0.0359,1.0000,1.0000
7,4,oot,60473,0.0359,0.0360,1.0022,0.9130
8,5,valid,60427,0.0446,0.0446,1.0000,1.0000
9,5,oot,61533,0.0446,0.0435,0.9738,0.1628


,value
relevel_factor_from_2017H1,0.975947
H2_pred_before,0.046392
H2_pred_after,0.045276
H2_actual,0.044587


## Cell 11 - Scorecard points table (WoE logistic) and coefficient checks

**Banker's note.** The classical benchmark is turned into a points scorecard (PDO 20, 600 points at 20:1 good:bad odds). A statsmodels Logit on the same WoE inputs gives p-values and a convergence flag; every coefficient on a WoE input should be positive (WoE = log bad/good, so a riskier bin raises the log-odds), which is what turns riskier bins into fewer points.

**Plain-English note.** The scorecard is the 'show your working' model: a table of points per applicant bin that a credit officer can read and add up by hand.

In [13]:
import statsmodels.api as sm, math
wm = models[WOE_NAME]; woe = wm.named_steps["woe"]; lr = wm.named_steps["clf"]
factor = L.SCALING["pdo"] / math.log(2.0)
rows = []
for j, col in enumerate(woe.columns_):
    for key, val in woe.maps_[col].items():
        rows.append({"feature": col, "bin": woe.bin_label(col, key), "woe": float(val), "points": float(-factor * lr.coef_[0][j] * val)})
pts = pd.DataFrame(rows).sort_values(["feature", "bin"]); save_table(pts, "points_table.csv")
Zs = woe.transform(xs); lg = sm.Logit(samp.bad_12m.to_numpy(float), sm.add_constant(Zs)).fit(disp=0, maxiter=200)
lg_tbl = pd.DataFrame({"coef": lg.params, "p_value": lg.pvalues, "odds_ratio": np.exp(lg.params)}); save_table(lg_tbl.reset_index().rename(columns={"index": "term"}), "statsmodels_logit_woe.csv")
display(lg_tbl.round(4))
M["scorecard"] = {"n_bins": int(len(pts)), "pdo": L.SCALING["pdo"], "base_score": L.SCALING["base_score"], "base_odds": L.SCALING["base_odds"],
                  "logit_converged": bool(lg.mle_retvals.get("converged", True)), "n_wrong_sign_coefs": int((lr.coef_[0] < 0).sum()), "wrong_sign_features": [c for c, b in zip(woe.columns_, lr.coef_[0]) if b < 0],
                  "n_coefs": int(lr.coef_.size), "n_pvalue_above_0.05": int((lg.pvalues.drop("const") > 0.05).sum())}
print(M["scorecard"])

,coef,p_value,odds_ratio
const,-3.2567,0.0000,0.0385
loan_amnt,0.9234,0.0001,2.5178
term_months,1.1041,0.0000,3.0164
emp_length_years,0.7227,0.0000,2.0600
annual_inc,0.5779,0.0000,1.7824
dti,0.8389,0.0000,2.3137
delinq_2yrs,-0.5693,0.3606,0.5659
inq_last_6mths,0.9461,0.0000,2.5757
open_acc,0.7046,0.0403,2.0230
pub_rec,-1.1725,0.0001,0.3096


{'n_bins': 187, 'pdo': 20, 'base_score': 600, 'base_odds': 20.0, 'logit_converged': True, 'n_wrong_sign_coefs': 3, 'wrong_sign_features': ['delinq_2yrs', 'pub_rec', 'application_type'], 'n_coefs': 19, 'n_pvalue_above_0.05': 3}


## Cell 12 - PD term structure (hazard and cumulative incidence)

**Banker's note.** Exposure-based monthly default hazard by month on book, separately for 36- and 60-month loans, on ALL vintages (loans leave the risk set when they prepay or when the file ends). Event time is the month a loan reaches 90 DPD (last payment month + 3). Cumulative incidence allows for prepayment as a competing risk; the default-only Kaplan-Meier curve is the 'nobody prepays' version. The 12-month point is cross-checked against the label rate. The old version computed the hazard on resolved loans only, which censored survivors and inflated lifetime PD.

**Plain-English note.** Default risk is not flat: it builds up over the first year or two and then fades. We measure that shape directly from how real loans behaved month by month.

In [14]:
base = df[df.mob >= 1]
haz = {t: L.hazard_by_month(base[base.term_months == t], max_m=84) for t in (36, 60)}
ts_rows = []
for t in (36, 60):
    h = haz[t]
    for hz in range(12, t + 1, 12):
        ts_rows.append({"term": t, "horizon_m": hz, "is_contractual_lifetime": hz == t, "cum_pd_incidence": float(h["cif"][hz]), "cum_pd_default_only": float(h["km"][hz])})
ts = pd.DataFrame(ts_rows); save_table(ts, "pd_term_structure.csv"); display(ts.round(4))
xchk = float(L.hazard_by_month(base[(base.split != "psi_only")], max_m=24)["cif"][12])
M["term_structure"] = {"table": ts.round(5).to_dict("records"), "pooled_cif_12m": round(xchk, 5), "label_rate_all_labelled": round(float(lab.bad_12m.mean()), 5),
                       "pool_p12_36": float(haz[36]["cif"][12]), "pool_p12_60": float(haz[60]["cif"][12]),
                       "prepaid_share_by_term": {t: float(haz[t]["prepaid"].sum() / haz[t]["at_risk"][1]) for t in (36, 60)},
                       "note": "hazard from all vintages (2018 vintages enter as censored exposure); lifetime = contractual term (36 or 60 months); months with <200 loans at risk get hazard 0"}
print("cross-check: pooled CIF at 12m =", round(xchk, 4), "| label bad rate (labelled loans) =", round(float(lab.bad_12m.mean()), 4))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for t, c in ((36, "#4C72B0"), (60, "#C44E52")):
    ax[0].plot(haz[t]["months"][1:t + 1], haz[t]["hazard"][1:t + 1], color=c, label=f"{t}m term"); ax[1].plot(haz[t]["months"][:t + 1], haz[t]["cif"][:t + 1], color=c, label=f"{t}m cumulative incidence")
    ax[1].plot(haz[t]["months"][:t + 1], haz[t]["km"][:t + 1], color=c, ls=":", label=f"{t}m default-only (no prepayment)")
ax[0].set_title("Monthly default hazard, months 1 to contractual term"); ax[1].set_title("Cumulative default probability to contractual term"); ax[0].legend(); ax[1].legend(fontsize=7); ax[0].set_xlabel("months on book"); ax[1].set_xlabel("months on book")
save_fig(fig, "pd_term_structure.png")
HAZ36, HAZ60 = haz[36]["hazard"].copy(), haz[60]["hazard"].copy()
HAZ36[37:] = HAZ36[36]; HAZ60[61:] = HAZ60[60]       # past-maturity (overdue) loans keep the end-of-term hazard
PRE36, PRE60 = haz[36]["prepay_hazard"].copy(), haz[60]["prepay_hazard"].copy()
PRE36[37:] = 0.0; PRE60[61:] = 0.0
POOL_P12 = {36: float(haz[36]["cif"][12]), 60: float(haz[60]["cif"][12])}   # same basis as the label: default within 12m, prepayment allowed

,term,horizon_m,is_contractual_lifetime,cum_pd_incidence,cum_pd_default_only
0,36,12,False,0.0387,0.0427
1,36,24,False,0.1061,0.1336
2,36,36,True,0.1483,0.2166
3,60,12,False,0.0546,0.0590
4,60,24,False,0.1518,0.1783
5,60,36,False,0.2202,0.2810
6,60,48,False,0.2648,0.3681
7,60,60,True,0.2877,0.4331


cross-check: pooled CIF at 12m = 0.0415 | label bad rate (labelled loans) = 0.0413


## Cell 13 - EAD: amortisation schedule validated against what was actually owed

**Banker's note.** For an instalment loan EAD is the scheduled outstanding principal when the loan defaults. The model is the level-payment amortisation formula; it is validated on charged-off loans against realised exposure at default (funded principal minus principal repaid before charge-off). **This validation (correlation and error) is an EAD result, not an LGD result.** Term loans carry no undrawn limit, so a credit-conversion factor does not apply.

**Plain-English note.** Before asking 'how much is lost', we ask 'how much was owed when it went wrong?'. For a normal loan that is just the remaining balance, and our balance formula reproduces it very closely.

In [15]:
co = L.lgd_frame(df)
co["ead_model"] = L.amortised_balance(co.funded_amnt, co.int_rate, co.installment, co.mtlp_f.clip(upper=co.term_months))
ok = co.ead_model.notna() & co.ead_obs.notna()
e = co[ok]
ead = {"n": int(len(e)), "corr_model_vs_observed": round(float(np.corrcoef(e.ead_model, e.ead_obs)[0, 1]), 5),
       "mae_usd": round(float(np.mean(np.abs(e.ead_model - e.ead_obs))), 2), "median_ratio_model_over_observed": round(float(np.median(e.ead_model / e.ead_obs)), 4),
       "mean_observed_ead": round(float(e.ead_obs.mean()), 2), "mean_ead_over_funded": round(float((e.ead_obs / e.funded_amnt).mean()), 4),
       "ccf": "not applicable: amortising term loans have no undrawn limit"}
M["ead"] = ead; display(pd.Series(ead).to_frame("value"))
fig, ax = plt.subplots(figsize=(5, 5)); sm_ = e.sample(min(20000, len(e)), random_state=1); ax.scatter(sm_.ead_obs, sm_.ead_model, s=2, alpha=0.3); mx = float(sm_.ead_obs.max())
ax.plot([0, mx], [0, mx], "k:"); ax.set_xlabel("observed EAD (funded - principal repaid)"); ax.set_ylabel("amortisation-formula EAD"); ax.set_title(f"EAD validation (r = {ead['corr_model_vs_observed']:.3f})"); save_fig(fig, "ead_validation.png")

,value
n,269291
corr_model_vs_observed,0.99342
mae_usd,278.49
median_ratio_model_over_observed,1.0
mean_observed_ead,11156.82
mean_ead_over_funded,0.6973
ccf,not applicable: amortising term loans have no ...


## Cell 14 - LGD: does any model beat the mean, out of time?

**Banker's note.** Realised workout LGD = 1 - (recoveries - collection fees) / EAD at default, floored at 0 and capped at 1, undiscounted (discounted variant shown as a sensitivity). The charge-off date is not published, so it is estimated as issue + months to last payment + 4. Only charge-offs with at least ~2 years of workout are used (estimated charge-off year <= 2016); later charge-offs have unfinished recoveries and would overstate LGD (shown below as the immature cohort). Time split on charge-off year: train <= 2014, valid 2015, out-of-time 2016. Candidates: training-mean benchmark, fractional logit, direct gradient boosting, and a two-stage model (P(any recovery) x expected recovery given recovery). A model is adopted only if it beats the mean on validation with a bootstrap interval excluding zero; its out-of-time skill is then reported whatever it is. Features are origination attributes plus EAD and time-to-default (known at default). LGD skill is never inferred from the EAD correlation.

**Plain-English note.** Loss given default is how much of the money is lost once someone has defaulted. We try to predict it from the borrower and the loan, and check honestly whether we beat just quoting the average. If we do not, we say so.

In [16]:
import lightgbm as lgb
import statsmodels.api as sm
mature = co[co.co_year <= 2016].copy()
immature = co[co.co_year >= 2017]
split_lgd = np.where(mature.co_year <= 2014, "train", np.where(mature.co_year == 2015, "valid", "oot"))
mature["lgd_split"] = split_lgd
imm = {"n": int(len(immature)), "mean_lgd": round(float(immature.lgd_obs.mean()), 4), "mean_lgd_mature": round(float(mature.lgd_obs.mean()), 4)}
cat_levels = {c: sorted(co[c].astype(str).unique()) for c in L.LGD_CAT}
def feats(frame):
    f = L.lgd_features(frame).copy()
    for c in L.LGD_CAT: f[c] = pd.Categorical(f[c].astype(str), categories=cat_levels[c])
    return f
F = {s: feats(mature[mature.lgd_split == s]) for s in ("train", "valid", "oot")}
Yl = {s: mature[mature.lgd_split == s].lgd_obs.to_numpy(float) for s in F}
RR = {s: mature[mature.lgd_split == s].rec_rate.to_numpy(float) for s in F}
ANY = {s: mature[mature.lgd_split == s].any_rec.to_numpy(int) for s in F}
base_mean = float(Yl["train"].mean())
tick("LGD sample sizes: " + ", ".join(f"{s}={len(Yl[s]):,}" for s in F) + f" | mean LGD train={base_mean:.4f}")

preds = {s: {"training mean": np.full(len(Yl[s]), base_mean)} for s in F}
# fractional logit (quasi-binomial) on standardised numerics + dummies
def design(f, ref=None):
    num = f[[c for c in L.LGD_NUM]].astype(float)
    if ref is None: ref = (num.median(), num.std().replace(0, 1))
    num = ((num.fillna(ref[0]) - ref[0]) / ref[1]).clip(-5, 5)
    dm = pd.get_dummies(f[["purpose", "home_ownership", "verification_status"]].astype(str), drop_first=True).astype(float)
    return num, dm, ref
num_tr, dm_tr, ref = design(F["train"]); cols_dm = list(dm_tr.columns)
Xtr = sm.add_constant(pd.concat([num_tr, dm_tr], axis=1))
glm = sm.GLM(Yl["train"], Xtr, family=sm.families.Binomial()).fit(maxiter=100)
for s in F:
    n_, d_, _ = design(F[s], ref); d_ = d_.reindex(columns=cols_dm, fill_value=0.0)
    preds[s]["fractional logit"] = glm.predict(sm.add_constant(pd.concat([n_, d_], axis=1), has_constant="add")).to_numpy()
# direct boosting
kw = dict(n_estimators=150 if FAST else 400, learning_rate=0.03, num_leaves=15, min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5.0, random_state=L.SEED, n_jobs=4, verbosity=-1)
reg = lgb.LGBMRegressor(**kw).fit(F["train"], Yl["train"])
for s in F: preds[s]["boosting (direct)"] = np.clip(reg.predict(F[s]), 0, 1)
# two-stage
clf = lgb.LGBMClassifier(**kw).fit(F["train"], ANY["train"])
pos = ANY["train"] == 1
reg2 = lgb.LGBMRegressor(**kw).fit(F["train"][pos], RR["train"][pos])
for s in F:
    preds[s]["two-stage (any recovery x amount)"] = np.clip(1.0 - clf.predict_proba(F[s])[:, 1] * np.clip(reg2.predict(F[s]), 0, 1), 0, 1)
auc_any = {s: float(roc_auc_score(ANY[s], clf.predict_proba(F[s])[:, 1])) for s in ("valid", "oot")}

lg_rows = []
for s in ("valid", "oot"):
    for name, p in preds[s].items():
        r = L.lgd_metrics(Yl[s], p, base_mean); r.update({"window": s, "model": name}); lg_rows.append(r)
lg = pd.DataFrame(lg_rows)[["window", "model", "n", "mean_actual", "mean_pred", "rmse", "mae", "rmse_mean_baseline", "skill_vs_mean", "spearman"]]
save_table(lg, "lgd_model_comparison.csv"); display(lg.round(4))
# rule: adopt only if valid bootstrap CI of RMSE gain excludes zero
adopt, boots = None, {}
for name in [k for k in preds["valid"] if k != "training mean"]:
    b = L.bootstrap_rmse_gain(Yl["valid"], preds["valid"][name], base_mean, n_iter=200 if FAST else 1000); boots[name] = b
best_name = min(boots, key=lambda k: -boots[k]["gain"])
if boots[best_name]["ci95"][0] > 0: adopt = best_name
oot_boot = {name: L.bootstrap_rmse_gain(Yl["oot"], preds["oot"][name], base_mean, n_iter=200 if FAST else 1000) for name in boots}
lgd_final_mean = float(mature.lgd_obs.mean())
verdict = (f"adopted '{adopt}' on validation" if adopt else "no model beat the training mean on validation (bootstrap CI includes zero)")
best_oot = max(oot_boot, key=lambda k: oot_boot[k]["gain"])
any_beats = any(v["ci95"][0] > 0 for v in oot_boot.values())
oot_verdict = ("a model beat the training mean out of time (bootstrap CI above zero)" if any_beats else
               "NO LGD model beat the training mean out of time (best candidate: RMSE gain %+.4f, 95%% CI %s)" % (oot_boot[best_oot]["gain"], oot_boot[best_oot]["ci95"]))
M["lgd"] = {"definition": "1 - (recoveries - collection fees)/EAD, clipped [0,1], undiscounted; mature charge-offs (est. charge-off year <= 2016)",
            "n_mature": int(len(mature)), "mean_lgd_mature": round(lgd_final_mean, 5), "median_lgd_mature": round(float(mature.lgd_obs.median()), 5),
            "share_lgd_one": round(float((mature.lgd_obs >= 0.9999).mean()), 4), "share_no_recovery": round(float((mature.any_rec == 0).mean()), 4),
            "discounted_mean_12m_lag_eir": round(float(mature.lgd_disc.mean()), 5), "immature_cohort": imm,
            "split_sizes": {s: int(len(Yl[s])) for s in F}, "train_mean": round(base_mean, 5),
            "models": lg.round(5).to_dict("records"), "validation_bootstrap_vs_mean": boots, "oot_bootstrap_vs_mean": oot_boot,
            "adopted_model": adopt, "verdict": verdict, "oot_verdict": oot_verdict, "any_model_beats_mean_oot": bool(any_beats), "best_oot_model_by_rmse_gain": best_oot,
            "oot_skill_best_model_vs_mean": oot_boot[best_oot], "auc_any_recovery": auc_any,
            "by_term": mature.groupby("term_months").lgd_obs.agg(["mean", "size"]).round(4).to_dict("index"),
            "note": "ECL uses the mature portfolio-mean LGD: the models need EAD-at-default and time-to-default, which an active loan does not have yet"}
print("LGD validation verdict:", verdict); print("LGD OOT verdict:", oot_verdict); print("best OOT model:", best_oot, oot_boot[best_oot]); print("immature cohort:", imm)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(mature.lgd_obs, bins=40, color="#4C72B0"); ax[0].set_title("Realised LGD (mature charge-offs)"); ax[0].set_xlabel("LGD")
pm = preds["oot"][best_oot]; q = pd.qcut(pd.Series(pm).rank(method="first"), 10, labels=False)
d = pd.DataFrame({"p": pm, "y": Yl["oot"], "q": q}).groupby("q").mean(); ax[1].plot(d.p, d.y, "o-", label=best_oot); ax[1].axhline(base_mean, color="grey", ls=":", label="training mean")
ax[1].set_xlabel("predicted LGD (decile mean)"); ax[1].set_ylabel("actual LGD (2016 charge-offs)"); ax[1].legend(fontsize=7); ax[1].set_title("Out-of-time LGD: predicted vs actual")
save_fig(fig, "lgd_study.png")

[   2782s] LGD sample sizes: train=21,575, valid=26,424, oot=54,429 | mean LGD train=0.9056


,window,model,n,mean_actual,mean_pred,rmse,mae,rmse_mean_baseline,skill_vs_mean,spearman
0,valid,training mean,26424,0.8834,0.9056,0.1073,0.0758,0.1073,0.0000,NaN
1,valid,fractional logit,26424,0.8834,0.9006,0.1072,0.0740,0.1073,0.0009,0.1421
2,valid,boosting (direct),26424,0.8834,0.8924,0.1056,0.0714,0.1073,0.0311,0.1468
3,valid,two-stage (any recovery x amount),26424,0.8834,0.8923,0.1056,0.0713,0.1073,0.0311,0.1483
4,oot,training mean,54429,0.9027,0.9056,0.1143,0.0673,0.1143,0.0000,NaN
5,oot,fractional logit,54429,0.9027,0.9031,0.1149,0.0696,0.1143,-0.0109,0.1340
6,oot,boosting (direct),54429,0.9027,0.8910,0.1151,0.0738,0.1143,-0.0148,0.1276
7,oot,two-stage (any recovery x amount),54429,0.9027,0.8906,0.1152,0.0741,0.1143,-0.0164,0.1270


LGD validation verdict: adopted 'boosting (direct)' on validation
LGD OOT verdict: NO LGD model beat the training mean out of time (best candidate: RMSE gain -0.0006, 95% CI [-0.00084, -0.000386])
best OOT model: fractional logit {'gain': -0.000619, 'ci95': [-0.00084, -0.000386], 'p_model_not_better': 1.0}
immature cohort: {'n': 166863, 'mean_lgd': 0.915, 'mean_lgd_mature': 0.8983}


## Cell 15 - The live book: refreshed risk, SICR test and IFRS 9 stage allocation

**Banker's note.** Reporting date = the file snapshot. The live book is every loan still active (Current, Late, Grace, Default) with an outstanding balance. SICR (significant increase in credit risk) is a relative-PD test: the 12-month PD re-scored with the refreshed FICO (`last_fico_range`, pulled ~Mar-2019) must be at least 2.0x the origination PD AND at least 1.0 percentage point higher, OR the loan is Late (31-120 days) (30-DPD backstop). Stage 3 (credit-impaired) = status `Default`, or Late (31-120) with at least 3 months since the last payment (a 90+ DPD proxy, the same rule that defines the PD target; LendingClub publishes no day count). Looser/stricter variants (2 months; all Late 31-120) are sensitivities. Everything else is Stage 1. Origination PD uses the frozen champion and the validation-fitted calibrator.

**Plain-English note.** Every loan still being repaid gets a health label. Stage 1 = fine, Stage 2 = noticeably riskier than when it was granted (so the bank must reserve for losses over the whole life, not just a year), Stage 3 = already in default.

In [17]:
act = df[df.loan_status.astype(str).isin(L.ACTIVE_STATUS) & (df.out_prncp > 0)].copy()
act["status"] = act.loan_status.astype(str)
Xa = act[L.PD_FEATURES].astype({c: "object" for c in L.CATEGORICAL})
champ = models[champion_name]
pd_orig = CAL.predict(champ.predict_proba(Xa)[:, 1])
Xn = Xa.copy(); has_new = act.last_fico_mid.notna().to_numpy(); Xn.loc[has_new, "fico_mid"] = act.last_fico_mid[has_new].to_numpy()
pd_now = CAL.predict(champ.predict_proba(Xn)[:, 1])
status = act.status.to_numpy()
missed = (SNAP_IDX - L.month_index(act.last_pymnt_d)).to_numpy()      # months since the last payment (90+ DPD proxy: >= 3, same rule as the PD label)
def assign(ratio=2.0, abs_pp=0.01, late_to_s3=False, backstop=True, s3_missed=3):
    flag, rel, back = L.sicr_flags(pd_orig, pd_now, status, ratio, abs_pp, backstop)
    st = np.ones(len(act), int); st[flag] = 2
    st[status == "Default"] = 3
    late = status == "Late (31-120 days)"
    st[late & (missed >= s3_missed)] = 3
    if late_to_s3: st[late] = 3
    return st, rel, back
stage_base, rel, back = assign()
stg = pd.DataFrame({"stage": stage_base, "status": status, "bal": act.out_prncp.to_numpy()})
tbl = stg.groupby("stage").agg(loans=("bal", "size"), exposure=("bal", "sum")); tbl["share_loans"] = tbl.loans / tbl.loans.sum(); tbl["share_exposure"] = tbl.exposure / tbl.exposure.sum()
display(tbl.round(4)); display(pd.crosstab(stg.status, stg.stage))
M["staging_inputs"] = {"n_active": int(len(act)), "exposure_usd": float(act.out_prncp.sum()), "share_with_refreshed_fico": float(has_new.mean()),
                       "sicr_rule": "PD_now >= 2.0 x PD_orig AND PD_now - PD_orig >= 1.0pp, OR Late(31-120) backstop", "stage3_rule": "status == Default OR (Late 31-120 AND >=3 months since last payment: 90+ DPD proxy, same rule as the PD target)",
                       "mean_pd_orig": float(pd_orig.mean()), "mean_pd_now": float(pd_now.mean()), "sicr_by_pd_test": int(rel.sum()), "sicr_by_backstop": int(back.sum())}
# sanity checks on the stage logic
chk = pd.DataFrame({"stage": stage_base, "pd_orig": pd_orig, "pd_now": pd_now, "ratio": pd_now / pd_orig})
print("sanity: mean PD ratio by stage"); display(chk.groupby("stage").agg(n=("ratio", "size"), mean_pd_now=("pd_now", "mean"), mean_ratio=("ratio", "mean")))
stage_checks = {"mean_refreshed_pd_rises_with_stage": bool(chk.groupby("stage").pd_now.mean().is_monotonic_increasing)}
print("stage checks:", stage_checks)
assert set(np.unique(stage_base)) <= {1, 2, 3}
assert (stage_base[status == "Default"] == 3).all() and (stage_base[(status == "Current") & ~rel] == 1).all()
assert (stage_base[(status == "Late (31-120 days)") & (missed >= 3)] == 3).all()

,loans,exposure,share_loans,share_exposure
stage,,,,
1,855966,8.946075e+09,0.9428,0.9407
2,42821,4.591990e+08,0.0472,0.0483
3,9117,1.051107e+08,0.0100,0.0111


stage,1,2,3
status,,,
Current,844434,29346,0
Default,0,0,40
In Grace Period,7669,660,0
Late (16-30 days),3863,451,0
Late (31-120 days),0,12364,9077


sanity: mean PD ratio by stage


,n,mean_pd_now,mean_ratio
stage,,,
1,855966,0.045124,1.096588
2,42821,0.073362,2.632139
3,9117,0.087488,1.583914


stage checks: {'mean_refreshed_pd_rises_with_stage': True}


## Cell 16 - Expected credit loss (ECL)

**Banker's note.** ECL = sum over future months of marginal PD x LGD x EAD x discount factor. Stage 1 uses a 12-month horizon, Stage 2 the remaining contractual life, Stage 3 is LGD x balance (PD = 1). Per-loan hazards scale the empirical term-structure hazard by a relative-risk multiplier so that the 12-month PD matches the loan's calibrated PD; EAD follows the scheduled balance; discounting is at the loan's own rate (EIR). LGD = mature portfolio mean (Cell 14: no model had demonstrated skill). The empirical prepayment hazard shortens expected life (a no-prepayment version is a sensitivity). Sensitivities are labelled as such - they are not forecasts and there are no macro scenario weights.

**Plain-English note.** We add up, month by month, the chance that each loan goes bad times the money that would then be owed times the share we would lose, in today's money. Loans that look riskier than at the start get a lifetime reserve; the rest a one-year reserve.

In [18]:
LGD_ECL = lgd_final_mean
term_a, mob_a = act.term_months.to_numpy(int), act.mob.to_numpy(int)
args = dict(term=term_a, mob=np.minimum(mob_a, 84), bal=act.out_prncp.to_numpy(), int_rate=act.int_rate.to_numpy(), installment=act.installment.to_numpy(),
            haz36=HAZ36, haz60=HAZ60, pool_p12=POOL_P12)
def run(stage_arr, lgd=LGD_ECL, pd_mult=1.0, pd_vec=None, prepay=True):
    pre = dict(pre36=PRE36, pre60=PRE60) if prepay else {}
    return L.project_ecl(stage=stage_arr, pd12=pd_now if pd_vec is None else pd_vec, lgd=lgd, pd_mult=pd_mult, **args, **pre)
ecl, pd_h = run(stage_base)
bal = act.out_prncp.to_numpy()
res = pd.DataFrame({"stage": stage_base, "ecl": ecl, "ead": bal, "pd_h": pd_h})
by = res.groupby("stage").agg(loans=("ead", "size"), ead=("ead", "sum"), ecl=("ecl", "sum"), mean_pd=("pd_h", "mean")); by["coverage"] = by.ecl / by.ead
by["share_loans"] = by.loans / by.loans.sum(); by["share_ead"] = by.ead / by.ead.sum(); by["share_ecl"] = by.ecl / by.ecl.sum()
by.loc["total"] = [by.loans.sum(), by.ead.sum(), by.ecl.sum(), np.nan, by.ecl.sum() / by.ead.sum(), 1, 1, 1]
save_table(by.reset_index(), "ecl_by_stage.csv"); display(by.round(4))
# identity check on Stage 1: ECL ~ PD12 x LGD x average discounted balance (loose bound)
s1 = res.stage == 1
approx = (res.pd_h[s1] * LGD_ECL * res.ead[s1]).sum()
chk = {"stage1_ecl": float(res.ecl[s1].sum()), "pd_x_lgd_x_opening_ead": float(approx), "ratio": float(res.ecl[s1].sum() / approx)}
print("identity bound (ECL <= PD x LGD x opening EAD):", chk)
assert chk["ratio"] <= 1.0001, "ECL cannot exceed PD x LGD x opening EAD"
cov = by.coverage.drop("total")
stage_checks["coverage_rises_with_stage"] = bool(cov.is_monotonic_increasing)
stage_checks["identity_ratio_in_0.5_1"] = bool(0.5 < chk["ratio"] <= 1.0001)
print("ECL checks:", stage_checks)
# sensitivities
sens = []
def add(name, e, st): sens.append({"scenario": name, "ecl": float(e.sum()), "coverage": float(e.sum() / bal.sum()), "stage2_share_loans": float((st == 2).mean()), "stage3_share_loans": float((st == 3).mean())})
add("base", ecl, stage_base)
for r_ in (1.5, 3.0):
    st_, _, _ = assign(ratio=r_); add(f"SICR ratio {r_}x", run(st_)[0], st_)
st_, _, _ = assign(s3_missed=2); add("Late(31-120) with >=2 months unpaid treated as Stage 3", run(st_)[0], st_)
st_, _, _ = assign(late_to_s3=True); add("all Late(31-120) treated as Stage 3", run(st_)[0], st_)
st_, _, _ = assign(ratio=1e9, backstop=False); add("no SICR (Stage 1 only + Default)", run(st_)[0], st_)
for m_ in (0.8, 1.2, 1.5): add(f"PD x{m_} (sensitivity)", run(stage_base, pd_mult=m_)[0], stage_base)
for l_ in (LGD_ECL - 0.05, LGD_ECL + 0.05): add(f"LGD {l_:.3f}", run(stage_base, lgd=l_)[0], stage_base)
add("LGD discounted (12m recovery lag)", run(stage_base, lgd=float(mature.lgd_disc.mean()))[0], stage_base)
add("PD at origination (no FICO refresh)", run(stage_base, pd_vec=pd_orig)[0], stage_base)
add("no prepayment credit (contractual survival)", run(stage_base, prepay=False)[0], stage_base)
sens = pd.DataFrame(sens); save_table(sens, "ecl_sensitivity.csv"); display(sens.round(4))
M["ecl"] = {"reporting_snapshot": snap_label, "n_loans": int(len(act)), "ead_usd": float(bal.sum()), "ecl_usd": float(ecl.sum()), "coverage": float(ecl.sum() / bal.sum()),
            "by_stage": {str(k): {kk: float(vv) for kk, vv in v.items()} for k, v in by.iterrows()}, "lgd_used": LGD_ECL, "identity_check": chk, "sanity_checks": stage_checks,
            "sensitivity": sens.round(5).to_dict("records"),
            "scope_notes": ["12m ECL for Stage 1, lifetime for Stage 2, LGD x balance for Stage 3", "no macro scenario weighting", "empirical prepayment hazard shortens expected life (sensitivity without it is in the table)",
                            "PD calibrated on the 2016 vintage; active-book vintages 2018 are scored with the same calibrator",
                            "model scored loans that were partly in its training window (survivors), so book PDs are mildly in-sample"]}
M["staging"] = {"shares_loans": {str(k): float(v) for k, v in tbl.share_loans.items()}, "shares_exposure": {str(k): float(v) for k, v in tbl.share_exposure.items()},
                "loans": {str(k): int(v) for k, v in tbl.loans.items()}, **M.pop("staging_inputs")}
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6)); b_ = by.drop("total")
ax[0].bar([f"Stage {i}" for i in b_.index], b_.share_loans * 100, color=["#55A868", "#DD8452", "#C44E52"][: len(b_)]); ax[0].set_title("Share of loans (%)")
ax[1].bar([f"Stage {i}" for i in b_.index], b_.coverage * 100, color=["#55A868", "#DD8452", "#C44E52"][: len(b_)]); ax[1].set_title("ECL coverage (% of exposure)"); save_fig(fig, "stage_ecl.png")

,loans,ead,ecl,mean_pd,coverage,share_loans,share_ead,share_ecl
stage,,,,,,,,
1,855966.0,8.946075e+09,5.649207e+08,0.0832,0.0631,0.9428,0.9407,0.7752
2,42821.0,4.591990e+08,6.936770e+07,0.2119,0.1511,0.0472,0.0483,0.0952
3,9117.0,1.051107e+08,9.442352e+07,1.0000,0.8983,0.0100,0.0111,0.1296
total,907904.0,9.510385e+09,7.287119e+08,NaN,0.0766,1.0000,1.0000,1.0000


identity bound (ECL <= PD x LGD x opening EAD): {'stage1_ecl': 564920658.4826072, 'pd_x_lgd_x_opening_ead': 734856477.2152106, 'ratio': 0.7687496483985732}
ECL checks: {'mean_refreshed_pd_rises_with_stage': True, 'coverage_rises_with_stage': True, 'identity_ratio_in_0.5_1': True}


,scenario,ecl,coverage,stage2_share_loans,stage3_share_loans
0,base,7.287119e+08,0.0766,0.0472,0.0100
1,SICR ratio 1.5x,7.550704e+08,0.0794,0.1145,0.0100
2,SICR ratio 3.0x,7.184512e+08,0.0755,0.0197,0.0100
3,Late(31-120) with >=2 months unpaid treated as...,7.879301e+08,0.0828,0.0393,0.0179
4,all Late(31-120) treated as Stage 3,8.306782e+08,0.0873,0.0335,0.0237
5,no SICR (Stage 1 only + Default),7.099613e+08,0.0747,0.0000,0.0100
6,PD x0.8 (sensitivity),6.076426e+08,0.0639,0.0472,0.0100
7,PD x1.2 (sensitivity),8.471096e+08,0.0891,0.0472,0.0100
8,PD x1.5 (sensitivity),1.020081e+09,0.1073,0.0472,0.0100
9,LGD 0.848,6.881524e+08,0.0724,0.0472,0.0100


## Cell 17 - Monitoring: population stability and vintage discrimination

**Banker's note.** PSI of the calibrated PD (bins from the training window) against validation, out-of-time and the 2018 vintages, CSI for the numeric features, and Gini/KS by issue quarter. Bands: < 0.10 stable, 0.10-0.25 watch, > 0.25 shift.

**Plain-English note.** Is the crowd applying today still the crowd the model learned from? PSI is a single number for how much the mix has moved.

In [19]:
pd_train = CAL.predict(models[champion_name].predict_proba(X["train"])[:, 1])
new18 = df[df.split == "psi_only"]
pd_18 = CAL.predict(models[champion_name].predict_proba(new18[L.PD_FEATURES].astype({c: "object" for c in L.CATEGORICAL}))[:, 1])
psi_rows = [{"comparison": "train -> valid 2016", "psi": L.psi(pd_train, pd_valid)}, {"comparison": "train -> oot 2017", "psi": L.psi(pd_train, pd_oot)},
            {"comparison": "train -> 2018 vintages", "psi": L.psi(pd_train, pd_18)}, {"comparison": "valid 2016 -> oot 2017", "psi": L.psi(pd_valid, pd_oot)}]
psi_t = pd.DataFrame(psi_rows); psi_t["band"] = psi_t.psi.apply(L.psi_band); save_table(psi_t, "psi_score.csv"); display(psi_t.round(4))
csi = [{"feature": c, "csi_oot": L.psi(pd.to_numeric(parts["train"][c], errors="coerce"), pd.to_numeric(parts["oot"][c], errors="coerce")),
        "csi_2018": L.psi(pd.to_numeric(parts["train"][c], errors="coerce"), pd.to_numeric(new18[c], errors="coerce"))} for c in L.NUMERIC]
csi = pd.DataFrame(csi).sort_values("csi_oot", ascending=False); csi["band_oot"] = csi.csi_oot.apply(L.psi_band); save_table(csi, "csi_features.csv"); display(csi.round(4))
vint = []
for s, p, y, praw in [("valid", parts["valid"], Y["valid"], p_valid_raw), ("oot", parts["oot"], Y["oot"], p_oot_raw)]:
    q = p.issue_d.dt.to_period("Q").astype(str).to_numpy()
    for qq in sorted(set(q)):
        m = q == qq
        if m.sum() > 500 and 0 < y[m].sum() < m.sum(): vint.append({"split": s, "issue_quarter": qq, "n": int(m.sum()), "bad_rate": float(y[m].mean()), "gini": L.gini(y[m], praw[m]), "ks": L.ks_stat(y[m], praw[m])})
vint = pd.DataFrame(vint); save_table(vint, "vintage_gini_ks.csv"); display(vint.round(4))
M["monitoring"] = {"psi_score": psi_t.round(5).to_dict("records"), "csi_top5": csi.head(5).round(5).to_dict("records"), "gini_by_quarter_range": [float(vint.gini.min()), float(vint.gini.max())]}
fig, ax = plt.subplots(figsize=(6, 3.4)); ax.bar(psi_t.comparison, psi_t.psi, color="#4C72B0"); ax.axhline(0.1, color="orange", ls="--"); ax.axhline(0.25, color="red", ls="--"); ax.set_xticklabels(psi_t.comparison, rotation=20, ha="right"); ax.set_title("Score PSI"); save_fig(fig, "psi_score.png")

,comparison,psi,band
0,train -> valid 2016,0.0013,stable
1,train -> oot 2017,0.0096,stable
2,train -> 2018 vintages,0.0309,stable
3,valid 2016 -> oot 2017,0.0042,stable


,feature,csi_oot,csi_2018,band_oot
10,revol_util,0.0937,0.2129,stable
6,inq_last_6mths,0.0371,0.0706,stable
12,mort_acc,0.0356,0.0513,stable
11,total_acc,0.0353,0.0707,stable
0,loan_amnt,0.0330,0.0580,stable
13,fico_mid,0.0275,0.1168,stable
9,revol_bal,0.0145,0.0302,stable
3,annual_inc,0.0115,0.0159,stable
2,emp_length_years,0.0104,0.0074,stable
7,open_acc,0.0085,0.0210,stable


,split,issue_quarter,n,bad_rate,gini,ks
0,valid,2016Q1,133887,0.0460,0.3905,0.2854
1,valid,2016Q2,97854,0.0478,0.3576,0.2627
2,valid,2016Q3,99120,0.0492,0.3438,0.2455
3,valid,2016Q4,103546,0.0456,0.3242,0.2318
4,oot,2017Q1,96779,0.0418,0.3538,0.2546
5,oot,2017Q2,105451,0.0446,0.3458,0.2510
6,oot,2017Q3,122701,0.0452,0.3519,0.2555
7,oot,2017Q4,118648,0.0440,0.3590,0.2620


## Cell 18 - Approval cut-off illustration (assumption-driven)

**Banker's note.** Illustration only, economics ASSUMED and stated: 12-month income = loan rate x amount, funding cost 4.5% p.a., operating cost USD 120 per loan, loss = default x LGD x amount (EAD = amount, no amortisation credit = conservative). The cut-off (minimum score) is chosen on **validation** realised outcomes and then evaluated unchanged on 2017. Profit uses realised defaults, not model PDs. The PD is not rescaled on the out-of-time sample.

**Plain-English note.** If we only lend to applicants above some score, how many loans do we give up and does the profit per loan improve? The numbers rest on stated assumptions, so treat them as a demonstration of the method, not a business case.

In [20]:
FUND, OPEX = 0.045, 120.0
def profit_curve(part, y, p, lgd=LGD_ECL):
    amt = part.loan_amnt.to_numpy(float); eir = part.int_rate.to_numpy(float) / 100
    unit = (eir - FUND) * amt - OPEX - y * lgd * amt
    order = np.argsort(p); out = []
    for frac in np.linspace(0.2, 1.0, 17):
        k = int(len(p) * frac); sel = order[:k]
        out.append({"approval_rate": frac, "max_pd": float(p[sel].max()), "profit_per_approved": float(unit[sel].mean()), "bad_rate": float(y[sel].mean()), "n": k})
    return pd.DataFrame(out)
cv_ = profit_curve(parts["valid"], Y["valid"], pd_valid); co_ = profit_curve(parts["oot"], Y["oot"], pd_oot)
best_valid = cv_.loc[cv_.profit_per_approved.idxmax()]
# apply the validation-chosen PD cut-off to 2017
appr = pd_oot <= best_valid.max_pd
amt = parts["oot"].loan_amnt.to_numpy(float); eir = parts["oot"].int_rate.to_numpy(float) / 100
unit = (eir - FUND) * amt - OPEX - Y["oot"] * LGD_ECL * amt
cut = {"cutoff_pd_chosen_on_valid": float(best_valid.max_pd), "oot_approval_rate": float(appr.mean()), "oot_profit_per_approved": float(unit[appr].mean()),
       "oot_profit_per_applicant_no_cutoff": float(unit.mean()), "oot_bad_rate_approved": float(Y["oot"][appr].mean()), "oot_bad_rate_all": float(Y["oot"].mean()),
       "assumptions": {"funding_cost": FUND, "opex_usd": OPEX, "lgd": LGD_ECL, "ead": "loan amount, 12-month window, no amortisation credit"}}
M["cutoff"] = cut; save_table(cv_, "cutoff_curve_valid.csv"); save_table(co_, "cutoff_curve_oot.csv"); display(pd.Series(cut).to_frame("value"))
fig, ax = plt.subplots(figsize=(6, 3.6)); ax.plot(cv_.approval_rate, cv_.profit_per_approved, label="valid 2016"); ax.plot(co_.approval_rate, co_.profit_per_approved, label="oot 2017")
ax.axvline(cut["oot_approval_rate"], color="grey", ls=":"); ax.set_xlabel("approval rate"); ax.set_ylabel("profit per approved loan (USD, assumed economics)"); ax.legend(); save_fig(fig, "cutoff_profit.png")

,value
cutoff_pd_chosen_on_valid,0.106719
oot_approval_rate,0.9559
oot_profit_per_approved,612.246948
oot_profit_per_applicant_no_cutoff,605.711024
oot_bad_rate_approved,0.040458
oot_bad_rate_all,0.043985
assumptions,"{'funding_cost': 0.045, 'opex_usd': 120.0, 'lg..."


## Cell 19 - Persist artifacts and the CV-facing numbers

**Banker's note.** Models, calibrator, hazards, tables and one `metrics.json` are written. The headline block (`cv`) is the single place the headline figures are read from; the README and model card quote it.

**Plain-English note.** Everything we quote is saved to files so nobody has to trust a number typed by hand.

In [21]:
import joblib, hashlib
joblib.dump({"champion": models[champion_name], "champion_name": champion_name, "woe_scorecard": models[WOE_NAME], "calibrator": CAL, "features": L.PD_FEATURES,
             "master_scale_edges": edges, "lgd_final_mean": LGD_ECL}, ART / "models" / "pd_models.joblib")
np.savez(ART / "models" / "hazards.npz", haz36=HAZ36, haz60=HAZ60)
ch, bm = M["pd"]["champion_oot"], M["pd"]["benchmark_oot"]; sg = M["pd"]["significance"].get("vs_woe_scorecard")
bo = bt.iloc[1]
ts_ = {(r["term"], r["horizon_m"]): r for r in M["term_structure"]["table"]}
M["cv"] = {
    "book": "LendingClub accepted loans 2007-2018Q4 (single dataset end to end)",
    "n_loans_in_file": M["data"]["rows_kept"], "n_labelled_12m": M["label"]["n_labelled"], "snapshot": snap_label,
    "champion": champion_name, "champion_oot_gini": ch["gini"], "champion_oot_ks": ch["ks"], "champion_oot_auc": ch["auc"],
    "woe_scorecard_oot_gini": bm["gini"], "woe_scorecard_oot_ks": bm["ks"],
    "champion_minus_woe_gini": sg["delong"]["delta_gini"] if sg else None,
    "champion_vs_woe_delong_p": sg["delong"]["p_value"] if sg else None,
    "champion_vs_woe_bootstrap_ci95": sg["bootstrap"]["ci95"] if sg else None,
    "champion_vs_runner_up_delong_p": (M["pd"]["significance"].get("vs_runner_up") or {}).get("delong", {}).get("p_value"),
    "calibration_method": cal_sel, "ece_oot": M["calibration"]["ece_oot_selected"],
    "backtest_predicted_oot": round(float(bo.predicted_pd), 5), "backtest_actual_oot": round(float(bo.actual_default_rate), 5),
    "backtest_ratio_oot": round(float(bo.ratio_actual_over_pred), 4), "backtest_within_10pct": M["backtest"]["oot_within_band"],
    "pd_12m_36m_loans": ts_[(36, 12)]["cum_pd_incidence"], "pd_lifetime_36m_loans": ts_[(36, 36)]["cum_pd_incidence"],
    "pd_12m_60m_loans": ts_[(60, 12)]["cum_pd_incidence"], "pd_lifetime_60m_loans": ts_[(60, 60)]["cum_pd_incidence"],
    "lgd_mean_mature": M["lgd"]["mean_lgd_mature"], "lgd_validation_verdict": M["lgd"]["verdict"], "lgd_oot_verdict": M["lgd"]["oot_verdict"], "lgd_any_model_beats_mean_oot": M["lgd"]["any_model_beats_mean_oot"], "lgd_oot_skill_vs_mean_rmse_gain": M["lgd"]["oot_skill_best_model_vs_mean"],
    "ead_amortisation_corr": M["ead"]["corr_model_vs_observed"], "ead_amortisation_mae_usd": M["ead"]["mae_usd"],
    "active_loans": M["ecl"]["n_loans"], "stage_shares_loans": M["staging"]["shares_loans"], "ead_usd": M["ecl"]["ead_usd"], "ecl_usd": M["ecl"]["ecl_usd"], "ecl_coverage": M["ecl"]["coverage"],
    "psi_oot": float(psi_t.psi.iloc[1]), "psi_2018": float(psi_t.psi.iloc[2]), "cutoff_oot_profit_per_approved": cut["oot_profit_per_approved"],
}
M["run"]["runtime_minutes"] = round((time.time() - T_START) / 60, 1)
M["run"]["versions"] = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__}
for mod in ("sklearn", "lightgbm", "xgboost", "catboost", "statsmodels", "scipy"):
    M["run"]["versions"][mod] = getattr(__import__(mod), "__version__", "?")
L.write_json(M, ART / "metrics.json")
L.write_json(M["cv"], ART / "cv_numbers.json")
import shutil; shutil.copy("rcl_lib.py", ART / "rcl_lib.py")
files = {str(p.relative_to(ART)): hashlib.sha256(p.read_bytes()).hexdigest()[:16] for p in sorted(ART.rglob("*")) if p.is_file() and p.name != "MANIFEST.json"}
L.write_json({"files": files, "runtime_minutes": M["run"]["runtime_minutes"]}, ART / "MANIFEST.json")
display(pd.Series(L.jsonable(M["cv"])).to_frame("value")); print("done in", M["run"]["runtime_minutes"], "min; artifacts:", len(files), "files")

,value
book,LendingClub accepted loans 2007-2018Q4 (single...
n_loans_in_file,2260668
n_labelled_12m,1765426
snapshot,2019-03
champion,CatBoost (tuned)
champion_oot_gini,0.35283
champion_oot_ks,0.25568
champion_oot_auc,0.67641
woe_scorecard_oot_gini,0.3134
woe_scorecard_oot_ks,0.22588


done in 48.1 min; artifacts: 47 files
